# h-BN AIMD: revised figures and interpretation

Place this notebook in `bvac-diff`, beside the existing `AIMD_results` folder, then run the cells in order in VS Code. `REPLOT_ONLY=True` reuses the saved coordinate caches instead of reading the large movies again. Corrected temperature/energy input CSVs are refreshed when the original metal folders are available. New outputs go to **`AIMD_results_updated`**; the original results are retained.

The requested style is set in one settings cell: line width **4**, tick/legend text **20 pt**, axis labels **26 pt**, and metal titles **26 pt**. Temperature limits are **200–500 K**. Energy-change limits are **−1 to +1 eV** for Au/Cu and **−3 to +1 eV** for Ag. The full numerical values remain in the exported CSVs; a range table reports values outside the displayed limits.

Au's temperature/energy source is explicitly **`6nn/gold/au-100ps/combined-temp-energy.csv`**. This override affects thermodynamics independently of cached coordinate data. `THERMO_SOURCE="csv"` and `REFRESH_THERMODYNAMICS=True` require the configured CSVs; a missing file stops the preflight instead of reusing previous exported thermodynamic data. The preflight prints the exact Au filename, row count, and first/last saved time values.

Migration is saved as **three separate XY figures**, with no embedded color bars, plus **one standalone Viridis time bar**. All three use the same time normalization. For these ~90–92 ps trajectories, the automatic common color scale is 0–100 ps. B and N sites retain green/blue static reference dots. Every figure is exported in PNG, SVG and PDF.

If you change atom selection, lattice, movie timing, or vacancy-neighbor definitions, set `REPLOT_ONLY=False` to read the original trajectory inputs. For styling changes leave it True. The notebook contains its analysis functions and needs no separate Python file.

Dependencies, if not already installed in the selected kernel:
```bash
python -m pip install numpy scipy pandas openpyxl matplotlib pillow ipykernel
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. The parent bvac-diff folder contains 6nn and 7nn.
# Use the current folder when VS Code/kernel starts in bvac-diff; otherwise
# use your known absolute path. Change this ONE path if the drive moves.
DATA_ROOT = Path.cwd()
if not ((DATA_ROOT / "6nn").is_dir() and (DATA_ROOT / "7nn").is_dir()):
    DATA_ROOT = Path(r"D:\pathfinder-cades\md-calc\bvac-diff")
# macOS example:   DATA_ROOT = Path("/Volumes/Research/hbn_aimd")

# 2. Choose the trajectory source. No upload is involved.
SOURCE = "xyz"                   # "xyz", "outcar", "csv", or "excel"
WORKBOOK = DATA_ROOT / "All_trajectory-md-90ps(1).xlsx"
if not WORKBOOK.exists() and (DATA_ROOT / "All_trajectory-md-90ps.xlsx").exists():
    WORKBOOK = DATA_ROOT / "All_trajectory-md-90ps.xlsx"

# 3. Map each metal to its actual folder and movie filename on your drive.
ACTIVE_METALS = ("Au", "Ag", "Cu")  # Use ("Au",) to check Au first.
FOLDERS = {"Au": Path("6nn") / "au-100ps",
           "Ag": Path("7nn") / "ag-100ps",
           "Cu": Path("7nn") / "cu-100ps"}
OUTPUT_PREFIXES = {"Au": "Au_bvac_6nn", "Ag": "Ag_bvac_7nn", "Cu": "Cu_bvac_7nn"}
COMPARISON_PREFIX = "_".join(ACTIVE_METALS) + "_bvac"
MOVIE_NAMES = {m: "movie-combined.xyz" for m in ACTIVE_METALS}
CSV_NAMES = {"Au": "Au_trajectory.csv", "Ag": "Ag_trajectory.csv", "Cu": "Cu_trajectory.csv"}
CSV_COLUMN_MAP = {}               # Canonical -> actual header; default time_ps,x_A,y_A,z_A.

# 4. Read temperature/energy independently of the coordinate movie.
THERMO_SOURCE = "csv"            # auto -> combined CSV, then OUTCAR; or csv/outcar/off.
THERMO_CSV_NAMES = {m: "combined-temp-energy.csv" for m in ACTIVE_METALS}
# Independent of the coordinate/movie folders: screenshot shows an extra gold directory.
THERMO_CSV_PATHS = {
    "Au": DATA_ROOT / "6nn" / "gold" / "au-100ps" / "combined-temp-energy.csv",
}
THERMO_COLUMN_MAP = {}            # e.g. {"time_ps":"Time", "temperature_K":"Temperature", "energy_eV":"Energy"}
THERMO_TIMESTEP_FS = None         # OUTCAR reads POTIM. Set 0.5 if a thermo CSV has only global Step.
THERMO_ENERGY_REFERENCE = "initial"  # "initial": E-E(0); "absolute": original energy values.
REFERENCE_TEMPERATURE_K = 300.0
SHOW_PART_FILE_TABLES = False
CHECK_RESTART_GEOMETRY = True
RESTART_GEOMETRY_TOL_A = 0.02

# XYZ usually lacks time metadata: specify the time BETWEEN SAVED FRAMES.
# If every MD step was saved, this is POTIM; if every 10th step was saved,
# use 10*POTIM. The spreadsheet spacing is 0.5 fs; verify your movie's spacing.
XYZ_FRAME_DT_FS = 0.5

# All atom indices in this notebook are ZERO-BASED (VESTA number minus one).
# "auto" looks for one triplet of undercoordinated N atoms in POSCAR.
# Inspect the proposed site before using its capture times in the manuscript.
# Use None for pristine systems, or specify the three N-neighbor indices.
VACANCY_NEIGHBORS = {"Au": "auto", "Ag": "auto", "Cu": "auto"}

# RUNS is built by the discovery cell after the functions cell. Numbered folders
# are ordered numerically; POSCAR comes from the earliest available run.
# Au uses au-100ps only. Restarts form one trajectory, not independent replicas.
SPLIT_RESTART_BOUNDARIES = True

# Styling-only rerun: reuse arrays in your existing AIMD_results/cache.
REPLOT_ONLY = True
PREVIOUS_RESULTS_DIR = DATA_ROOT / "AIMD_results"
REPLOT_RAW_CACHE = {}             # Only needed if several raw caches exist for one metal.
REFRESH_THERMODYNAMICS = True      # Reads corrected input CSV/OUTCARs when available.
OUTPUT_DIR = DATA_ROOT / "AIMD_results_updated"
USE_CACHE = True                    # Reuse extracted arrays if source files/settings match.
EXPORT_CSV = False                 # Original full coordinate CSVs are already saved.
SAVE_FIGURES = True
SHOW_FIGURES = True
SHOW_INDIVIDUAL_FIGURES = False     # Individual files are still saved.
FIGURE_FORMATS = ("png", "svg", "pdf")
FIGURE_DPI = 350
SAVE_INDIVIDUAL_METALS = True
SAVE_COMPARISON_FIGURES = True

# Exact font sizes in points; change these values for every figure at once.
FONT_SCALE = 1.0
FONT_SIZES = {"base": 20, "axes": 26, "ticks": 20, "title": 26, "legend": 20}
LINE_WIDTH = 4.0                  # Data/reference lines, axes, and ticks.
TEMPERATURE_YLIM = (200.0, 500.0)
ENERGY_YLIMS = {"Au": (-1.0, 1.0), "Ag": (-3.0, 1.0), "Cu": (-1.0, 1.0)}
MIGRATION_SAVE_COMPARISON = False  # Three separate migration figures by default.
MIGRATION_TIME_LIMITS_PS = None    # Shared range: 0 to next multiple of 10 ps.
MIGRATION_INCLUDE_COLORBAR = False # One standalone time bar, not one in each panel.

# Migration figure: actual reference B/N coordinates, tiled across periodic images.
SHOW_LATTICE = True
LATTICE_COLORS = {"B": "#249B59", "N": "#3659C9"}  # B green, N blue.
LATTICE_ALPHA = 0.40
LATTICE_DOT_SIZE = 25
PATH_CMAP = "viridis"               # Color indicates time; try "plasma" or "viridis".
PATH_LINE_WIDTH = LINE_WIDTH
MIGRATION_PADDING_A = 2.5

# Flag unusually large single-frame motion; do not smooth away or reconnect it.
JUMP_SPEED_A_PER_PS = 50.0

# MSD settings: compute on ONE continuous segment of each run.
# By default choose the longest segment; its exact interval is reported below.
ANALYSIS_WINDOWS_PS = {}            # Example after inspection: {"Au": (5.0, 60.0)}
SEGMENT_IDS = {}                    # Example: {"Au": 0}; otherwise longest valid segment.
MSD_STRIDE = 10                     # Decimate AFTER unwrapping; 10*0.5 fs = 5 fs.
MSD_MAX_LAG_PS = 10.0
COMPARE_POST_CAPTURE_MSD = True
POST_CAPTURE_WINDOW_PS = (12.0, 50.0)  # One continuous after-capture interval shared by all metals.

# Fits are deliberately opt-in after inspecting the stationary time window/MSD.
# Values are LAG-time limits, not absolute simulation times.
DIFFUSION_FIT_LAG_PS = {}            # Example: {"Au": (1.0, 3.0)}
DIFFUSION_BLOCKS = 4
CHARACTERISTIC_LENGTH_A = 5.0        # L for tau_D(L)=L^2/(4D), if a D fit is enabled.

CAPTURE_RADIUS_A = 1.5              # Lateral distance from the vacancy center.
CAPTURE_DWELL_PS = 0.5              # Persistent entry, not one instantaneous touch.
PLANE_CROSSING_DEADBAND_A = 0.25
PLANE_CROSSING_DWELL_PS = 0.10
SHOW_CAPTURE_SENSITIVITY = True

# Optional, leave None for the first run. Example: ANIMATE_LABEL = "Cu"
ANIMATE_LABEL = None
SAVE_ANIMATION_GIF = False

## Analysis functions

Run this cell once; it can stay collapsed afterward. It streams large text files, reads POSCAR, unwraps the periodic in-plane coordinates, removes B/N substrate translation when all atom coordinates exist, and keeps discontinuities out of kinetic calculations. Variable-cell MD and changes of atom ordering are rejected rather than analyzed with a fixed-cell assumption.

In [ ]:
"""Local h-BN AIMD postprocessing. Read-only inputs; results go to output_dir.

All distances are Angstrom and all times are ps. No ASE or VASP license is
needed to read an existing OUTCAR. The notebook embeds this entire module.
"""

from pathlib import Path
from itertools import product
import gzip
import bz2
import lzma
import re
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from scipy.signal import fftconvolve
from scipy.stats import linregress

COLORS = {"Au": "#C49A08", "Ag": "#85898F", "Cu": "#B86C32"}
FLOAT = r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[EeDd][-+]?\d+)?"


def number(text):
    return float(text.replace("D", "E").replace("d", "e"))


def minimum_image(displacements, cell, pbc=(True, True, False)):
    """Nearest image for a conventional slab cell, including a hexagonal cell.

    Search the neighboring lattice translations after fractional rounding;
    fractional rounding alone is not a nearest-image rule for skew cells.
    """
    a = np.asarray(displacements, float)
    if cell is None:
        return a.copy()
    original_shape = a.shape
    matrix = np.asarray(cell, float)
    f = a.reshape(-1, 3) @ np.linalg.inv(matrix)
    periodic = np.array(pbc, bool)
    f[:, periodic] -= np.rint(f[:, periodic])
    best = f @ matrix
    best2 = np.sum(best * best, axis=1)
    choices = [(-1, 0, 1) if flag else (0,) for flag in periodic]
    for shift in product(*choices):
        if not any(shift):
            continue
        candidate = (f + np.asarray(shift)) @ matrix
        d2 = np.sum(candidate * candidate, axis=1)
        improve = d2 < best2
        best[improve] = candidate[improve]
        best2[improve] = d2[improve]
    return best.reshape(original_shape)


def unwrap_positions(positions, cell, pbc=(True, True, False)):
    positions = np.asarray(positions, float)
    if cell is None or len(positions) < 2:
        return positions.copy()
    steps = minimum_image(np.diff(positions, axis=0), cell, pbc)
    return np.vstack([positions[0], positions[0] + np.cumsum(steps, axis=0)])


def text_file(path):
    path = Path(path)
    opener = {".gz": gzip.open, ".bz2": bz2.open, ".xz": lzma.open}.get(path.suffix, open)
    return opener(path, "rt", encoding="utf-8", errors="replace")


def read_poscar(path):
    """Read a VASP-5/6 POSCAR, including Cartesian/Direct/selective dynamics."""
    with text_file(path) as f:
        title = next(f).strip()
        scales = np.array([number(v) for v in next(f).split()])
        base = np.array([[number(v) for v in next(f).split()[:3]] for _ in range(3)])
        if len(scales) == 1:
            factor = scales[0]
            if factor < 0:
                factor = (abs(factor)/abs(np.linalg.det(base)))**(1/3)
            scale_xyz = np.repeat(factor, 3)
        elif len(scales) == 3 and np.all(scales > 0):
            scale_xyz = scales
        else:
            raise ValueError("Unsupported POSCAR scaling factors.")
        cell = base * scale_xyz[None, :]
        species = next(f).split()
        if all(v.isdigit() for v in species):
            raise ValueError("This VASP-4 POSCAR lacks element names; use a VASP-5/6 POSCAR.")
        counts = [int(v) for v in next(f).split()]
        if len(counts) != len(species):
            raise ValueError("POSCAR element/count mismatch.")
        line = next(f).strip()
        if line.lower().startswith("s"):
            line = next(f).strip()
        direct = line.lower().startswith("d")
        if not direct and not line.lower().startswith(("c", "k")):
            raise ValueError("Expected Direct or Cartesian POSCAR coordinates.")
        positions = np.array([[number(v) for v in next(f).split()[:3]] for _ in range(sum(counts))])
        positions = positions @ cell if direct else positions * scale_xyz
    return {"title": title, "cell": cell, "symbols": np.repeat(species, counts),
            "positions_A": positions}


def attach_structure(raw, structure, vacancy_neighbors=None):
    """Supply a static cell/reference plane to spreadsheet or metal-only XYZ data."""
    if structure is None:
        return raw
    raw = dict(raw)
    cell, snapshot, symbols = structure["cell"], structure["positions_A"], structure["symbols"]
    if raw.get("cell") is not None and not np.allclose(raw["cell"], cell, atol=1e-5):
        raise ValueError("POSCAR cell differs from trajectory cell; verify the folder mapping.")
    if raw.get("snapshot_A") is not None:
        if not np.array_equal(raw["symbols"], symbols):
            raise ValueError("POSCAR and trajectory element ordering differ; verify the selected POSCAR.")
        offset = np.linalg.norm(minimum_image(raw["snapshot_A"]-snapshot, cell), axis=1).max()
        raw["first_frame_poscar_max_offset_A"] = float(offset)
        if offset > 0.25:
            warnings.warn("First trajectory frame differs from the chosen POSCAR by more than 0.25 A; "
                          "check the initial run and N-neighbor indices.")
    raw["cell"] = cell
    raw.setdefault("snapshot_A", snapshot)
    raw.setdefault("symbols", symbols)
    normal = np.cross(cell[0], cell[1]); normal /= np.linalg.norm(normal)
    e1 = cell[0]/np.linalg.norm(cell[0])
    raw.setdefault("basis", np.vstack([e1, np.cross(normal, e1), normal]))
    if raw.get("height_A") is None:
        center = snapshot[np.isin(symbols, ("B", "N"))].mean(axis=0)
        raw["height_A"] = (raw["positions_A"]-center) @ normal
        raw["height_reference"] = "static POSCAR plane"
    if vacancy_neighbors is not None and raw.get("vacancy_centers_A") is None:
        ids = np.asarray(vacancy_neighbors, int)
        tri = snapshot[ids].copy()
        tri[1:] = tri[0]+minimum_image(tri[1:]-tri[0], cell)
        raw["vacancy_centers_A"] = np.tile(tri.mean(axis=0), (len(raw["time_ps"]), 1))
        raw["vacancy_reference"] = "static POSCAR N-neighbor centroid"
    return raw


def read_xyz_movie(path, metal, structure=None, metal_index=None, frame_dt_fs=0.5,
                   vacancy_neighbors=None, xyz_time_unit=None, progress_every=25000):
    """Stream a multi-frame XYZ/extXYZ movie; retain selected coordinates only.

    Standard XYZ: integer atom count, one comment line, then 'Element x y z'.
    A POSCAR supplies the cell when ordinary XYZ has no Lattice metadata.
    frame_dt_fs is the time BETWEEN SAVED FRAMES, not necessarily POTIM.
    Time= comments are used only when xyz_time_unit is explicitly 'fs' or 'ps'.
    """
    if frame_dt_fs is None or frame_dt_fs <= 0:
        raise ValueError("Set frame_dt_fs to the spacing between saved XYZ frames.")
    t, positions, drifts, heights, centers, normals = [], [], [], [], [], []
    cell = None if structure is None else np.asarray(structure["cell"])
    symbols0 = first = previous_full = full_u = initial_center = basis = None
    selected = metal_index
    previous_comment_time = None
    with text_file(path) as f:
        while True:
            line = f.readline()
            if not line:
                break
            if not line.strip():
                continue
            try:
                n = int(line.strip())
            except ValueError as exc:
                raise ValueError("Expected XYZ atom count. Select an .xyz/.extxyz coordinate movie.") from exc
            comment = f.readline()
            if not comment:
                warnings.warn("Ignored incomplete final XYZ frame.")
                break
            lattice = re.search(r'Lattice="([^"]+)"', comment)
            if lattice:
                lattice_cell = np.array([number(v) for v in lattice.group(1).split()]).reshape(3, 3)
                if cell is None:
                    cell = lattice_cell
                elif not np.allclose(cell, lattice_cell, atol=1e-5):
                    raise ValueError("XYZ cell changed; variable-cell trajectories need a different analysis.")
            props = re.search(r"Properties=([^\s]+)", comment)
            species_col, pos_col = 0, 1
            if props:
                entries, offset = props.group(1).split(":"), 0
                for j in range(0, len(entries), 3):
                    name, count = entries[j], int(entries[j+2])
                    if name in ("species", "element"):
                        species_col = offset
                    if name in ("pos", "positions"):
                        pos_col = offset
                    offset += count
            rows, symbols = [], []
            for _ in range(n):
                row = f.readline().split()
                if len(row) < pos_col+3:
                    break
                symbols.append(row[species_col])
                rows.append([number(v) for v in row[pos_col:pos_col+3]])
            if len(rows) != n:
                warnings.warn("Ignored incomplete final XYZ frame.")
                break
            full = np.array(rows)
            symbols = np.array(symbols)
            substrate = np.isin(symbols, ("B", "N"))
            if first is None:
                first, symbols0 = full.copy(), symbols.copy()
                idx = np.flatnonzero(symbols == metal)
                if selected is None:
                    if len(idx) != 1:
                        raise ValueError(f"Found {len(idx)} {metal} atoms; set metal_index.")
                    selected = int(idx[0])
                if not 0 <= selected < n or symbols[selected] != metal:
                    raise ValueError("metal_index does not identify the requested element.")
                if cell is None and np.any(substrate):
                    raise ValueError("Provide POSCAR or extXYZ Lattice metadata for full-atom XYZ.")
                normal = np.array([0., 0., 1.]) if cell is None else np.cross(cell[0], cell[1])
                normal /= np.linalg.norm(normal)
                e1 = np.array([1., 0., 0.]) if cell is None else cell[0]/np.linalg.norm(cell[0])
                basis = np.vstack([e1, np.cross(normal, e1), normal])
                full_u = full.copy()
                if np.any(substrate):
                    initial_center = full[substrate].mean(axis=0)
            else:
                if not np.array_equal(symbols, symbols0):
                    raise ValueError("Atom ordering or atom count changed within the XYZ movie.")
                if cell is None:
                    full_u += full-previous_full
                else:
                    step = (full-previous_full) @ np.linalg.inv(cell)
                    step[:, :2] -= np.rint(step[:, :2])
                    full_u += step @ cell
            comment_time = re.search(r"\b[Tt]ime\s*=\s*("+FLOAT+")", comment)
            if comment_time and xyz_time_unit in ("fs", "ps"):
                value = number(comment_time.group(1))*(0.001 if xyz_time_unit == "fs" else 1)
                if not t:
                    first_comment_time = value
                if previous_comment_time is not None and value <= previous_comment_time:
                    raise ValueError("XYZ comment times reset; split the restarted movies.")
                time = value-first_comment_time
                previous_comment_time = value
            else:
                time = len(t)*frame_dt_fs/1000
            t.append(time)
            positions.append(full[selected].copy())
            if np.any(substrate):
                center = full_u[substrate].mean(axis=0)
                drifts.append(center-initial_center)
                heights.append(np.dot(full_u[selected]-center, basis[2]))
            if vacancy_neighbors is not None and len(full) > 1:
                ids = np.asarray(vacancy_neighbors, int)
                if len(ids) != 3 or np.any(symbols[ids] != "N"):
                    raise ValueError("Vacancy neighbors must be three N indices in the XYZ ordering.")
                tri = full[ids].copy()
                tri[1:] = tri[0]+minimum_image(tri[1:]-tri[0], cell)
                normal = np.cross(tri[1]-tri[0], tri[2]-tri[0])
                normal /= np.linalg.norm(normal)
                if np.dot(normal, basis[2]) < 0:
                    normal = -normal
                centers.append(tri.mean(axis=0))
                normals.append(normal)
            previous_full = full.copy()
            if progress_every and len(t) % progress_every == 0:
                print(f"Read {len(t):,} XYZ frames ({time:.3f} ps)")
    if not t:
        raise ValueError("No complete XYZ movie frames read.")
    raw = {"time_ps": np.array(t), "positions_A": np.array(positions), "cell": cell,
           "metal": metal, "source": str(path), "basis": basis, "metal_index": selected}
    if len(first) > 1:
        raw.update(symbols=symbols0, snapshot_A=first)
    if drifts:
        raw.update(substrate_drift_A=np.array(drifts), height_A=np.array(heights),
                   height_reference="instantaneous B/N plane")
    if centers:
        raw.update(vacancy_centers_A=np.array(centers), vacancy_normals=np.array(normals),
                   vacancy_reference="instantaneous three-N centroid")
    return attach_structure(raw, structure, vacancy_neighbors)


def read_excel_trajectory(workbook, sheet, metal, cache_dir=None):
    path = Path(workbook).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Set WORKBOOK to your local Excel file: {path}")
    cache = None
    if cache_dir is not None:
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        key = f"{path}|{path.stat().st_size}|{path.stat().st_mtime_ns}|{sheet}"
        cache = cache_dir / (hashlib.sha256(key.encode()).hexdigest()[:18] + ".npz")
    if cache is not None and cache.exists():
        with np.load(cache, allow_pickle=False) as z:
            time, positions = z["time_ps"], z["positions_A"]
    else:
        df = pd.read_excel(path, sheet_name=sheet, engine="openpyxl")
        required = ["time_ps", "x_A", "y_A", "z_A"]
        if not set(required).issubset(df.columns):
            raise ValueError(f"{sheet}: expected columns {required}; found {list(df.columns)}")
        time = df["time_ps"].to_numpy(float)
        positions = df[["x_A", "y_A", "z_A"]].to_numpy(float)
        if cache is not None:
            np.savez_compressed(cache, time_ps=time, positions_A=positions)
    return {"time_ps": time, "positions_A": positions, "metal": metal,
            "cell": None, "source": str(path), "sheet": sheet}


def read_csv_trajectory(path, metal, column_map=None):
    """Read a combined metal-coordinate CSV, with explicit ps/Angstrom columns."""
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.strip()
    mapping = {name: name for name in ("time_ps", "x_A", "y_A", "z_A")}
    mapping.update(column_map or {})
    missing = [name for name, original in mapping.items() if original not in frame.columns]
    if missing:
        raise ValueError(f"{path}: expected columns {mapping}; found {list(frame.columns)}. "
                         "Set CSV_COLUMN_MAP explicitly if your headers differ.")
    return {"time_ps": frame[mapping["time_ps"]].to_numpy(float),
            "positions_A": frame[[mapping[v] for v in ("x_A", "y_A", "z_A")]].to_numpy(float),
            "metal": metal, "cell": None, "source": str(path)}


def read_thermodynamics_csv(path, column_map=None, timestep_fs=None):
    """Read a combined thermo CSV, keeping its own time axis and energy meaning.

    Mapping keys: time_ps OR time_fs OR step, temperature_K, energy_eV.
    Generic 'Energy' is retained as an unspecified energy, never called TOTEN.
    Step-based input requires an explicit MD timestep and nonresetting steps.
    """
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.strip()
    norm = lambda value: re.sub(r"[^a-z0-9]", "", str(value).lower())
    columns = {norm(c): c for c in frame.columns}
    mapping = dict(column_map or {})
    aliases = {
        "time_ps": ("time_ps", "time (ps)", "global_time_ps", "cumulative_time_ps"),
        "time_fs": ("time_fs", "time (fs)", "global_time_fs"),
        "step": ("global_step", "global_md_step", "cumulative_step", "step", "md_step", "ionic_step"),
        "temperature_K": ("temperature_K", "temperature (K)", "temperature", "temp_K", "temp", "T(K)", "T"),
        "energy_eV": ("TOTEN", "free_energy_eV", "potential_energy_eV", "potential energy (eV)",
                      "energy_eV", "energy (eV)", "energy", "total_energy_eV", "total_energy"),
    }
    for key, candidates in aliases.items():
        if key not in mapping:
            mapping[key] = next((columns[norm(c)] for c in candidates if norm(c) in columns), None)
    for key in ("temperature_K", "energy_eV"):
        if mapping[key] is None or mapping[key] not in frame:
            raise ValueError(f"{path}: cannot identify {key}; headers are {list(frame.columns)}. "
                             "Set THERMO_COLUMN_MAP explicitly, or use THERMO_SOURCE='outcar'.")
    explicit_times = [key for key in ("time_ps", "time_fs", "step") if key in (column_map or {})]
    time_key = explicit_times[0] if explicit_times else next(
        (key for key in ("time_ps", "time_fs", "step") if mapping[key] is not None), None)
    if time_key is None or mapping[time_key] not in frame:
        raise ValueError(f"{path}: supply a ps/fs time column or a global MD-step column. "
                         "A generic 'Time' column needs an explicit unit mapping.")
    time = pd.to_numeric(frame[mapping[time_key]], errors="coerce").to_numpy(float)
    if time_key == "time_fs":
        time *= 0.001
    elif time_key == "step":
        if timestep_fs is None or timestep_fs <= 0:
            raise ValueError("A step-based thermo CSV requires THERMO_TIMESTEP_FS.")
        time *= timestep_fs / 1000
    if len(time) < 2 or not np.all(np.isfinite(time)) or np.any(np.diff(time) <= 0):
        raise ValueError(f"{path}: thermodynamic times/steps must increase globally. "
                         "Use ordered OUTCARs if this CSV resets at each restart.")
    time -= time[0]
    temperature = pd.to_numeric(frame[mapping["temperature_K"]], errors="coerce").to_numpy(float)
    energy = pd.to_numeric(frame[mapping["energy_eV"]], errors="coerce").to_numpy(float)
    if not np.isfinite(temperature).any() or not np.isfinite(energy).any():
        raise ValueError(f"{path}: no numeric temperature or energy values found.")
    energy_header = str(mapping["energy_eV"])
    quantity = "TOTEN" if "toten" in norm(energy_header) else energy_header
    return {"time_ps": time, "temperature_K": temperature, "energy_eV": energy,
            "source_ids": np.zeros(len(time), int), "source": str(path),
            "energy_quantity": quantity, "column_mapping": mapping}


def read_outcar_thermodynamics(paths, timestep_fs=None):
    """Stream ionic-step temperature and electronic free energy TOTEN only.

    Coordinates are not retained. Restart files must already be in numeric order.
    Time at each file boundary advances by one POTIM; no missing elapsed time
    can be inferred from a concatenation. File boundaries remain in source_ids.
    """
    times, temperatures, energies, source_ids, ionic_steps = [], [], [], [], []
    paths = [Path(p) for p in paths]
    iteration_re = re.compile(r"Iteration\s+(\d+)\s*\(\s*\d+\s*\)")
    potim_re = re.compile(r"POTIM\s*=\s*(" + FLOAT + ")")
    energy_re = re.compile(r"free\s+energy\s+TOTEN\s*=\s*(" + FLOAT + ")")
    temp_re = re.compile(r"temperature\s*=?\s*(" + FLOAT + r")\s*K", re.I)
    for source_id, path in enumerate(paths):
        dt_fs, step, record, previous_step = timestep_fs, None, {}, None
        emitted = 0
        def emit(rec):
            nonlocal previous_step, emitted
            if not rec.get("has_position") or not ("temperature" in rec or "energy" in rec):
                return
            if dt_fs is None or dt_fs <= 0:
                raise ValueError(f"{path}: missing POTIM; set the thermo timestep explicitly.")
            current = rec.get("step")
            delta = current - previous_step if emitted and current is not None and previous_step is not None else 1
            if delta <= 0:
                raise ValueError(f"{path}: ionic-step numbering resets within this file.")
            times.append(0.0 if not times else times[-1] + delta*dt_fs/1000)
            temperatures.append(rec.get("temperature", np.nan))
            energies.append(rec.get("energy", np.nan))
            source_ids.append(source_id)
            ionic_steps.append(current if current is not None else np.nan)
            previous_step, emitted = current, emitted + 1
        with text_file(path) as handle:
            for line in handle:
                if "POTIM" in line and timestep_fs is None:
                    match = potim_re.search(line)
                    if match:
                        dt_fs = number(match.group(1))
                if "IBRION" in line:
                    match = re.search(r"IBRION\s*=\s*(-?\d+)", line)
                    if match and int(match.group(1)) != 0:
                        raise ValueError(f"{path}: IBRION is not zero; expected an MD OUTCAR.")
                if "Iteration" in line:
                    match = iteration_re.search(line)
                    if match:
                        current = int(match.group(1))
                        if step is not None and current != step:
                            emit(record)
                            record = {}
                        step = current
                        record["step"] = step
                if "POSITION" in line and "TOTAL-FORCE" in line:
                    if record.get("has_position"):
                        emit(record)
                        record = {"step": step}
                    record["has_position"] = True
                if "TOTEN" in line:
                    match = energy_re.search(line)
                    if match:
                        record["energy"] = number(match.group(1))
                if "temperature" in line.lower():
                    match = temp_re.search(line)
                    if match:
                        record["temperature"] = number(match.group(1))
            emit(record)
        if not emitted:
            raise ValueError(f"{path}: no ionic MD thermodynamics found.")
    if not times or not np.isfinite(temperatures).any() or not np.isfinite(energies).any():
        raise ValueError("No usable MD temperature and TOTEN values were found.")
    return {"time_ps": np.array(times), "temperature_K": np.array(temperatures),
            "energy_eV": np.array(energies), "source_ids": np.array(source_ids),
            "ionic_steps": np.array(ionic_steps), "source": [str(p) for p in paths],
            "energy_quantity": "TOTEN: electronic free energy"}


def load_thermodynamics(cfg, trajectory=None, source="auto", cache_dir=None, use_cache=True):
    """Independent thermo input: existing OUTCAR trajectory, combined CSV, or OUTCARs."""
    if source == "off":
        return None
    if source not in ("auto", "csv", "outcar"):
        raise ValueError("THERMO_SOURCE must be auto, csv, outcar, or off.")
    if (source in ("auto", "outcar") and cfg["kind"] == "outcar" and trajectory is not None
            and np.isfinite(trajectory.get("temperature_K", [np.nan])).any()
            and np.isfinite(trajectory.get("energy_eV", [np.nan])).any()):
        return {key: trajectory[key] for key in ("time_ps", "temperature_K", "energy_eV", "source_ids", "source")} | {
            "label": cfg["label"], "metal": cfg["metal"], "energy_quantity": "TOTEN: electronic free energy"}
    candidates = ("csv", "outcar") if source == "auto" else (source,)
    errors = []
    for kind in candidates:
        try:
            paths = [Path(cfg["thermo_csv"])] if kind == "csv" else [Path(p) for p in cfg["outcars"]]
            if not paths or any(not p.is_file() for p in paths):
                raise FileNotFoundError(f"{kind} input not available")
            if kind == "outcar" and cfg.get("missing_outcar_runs"):
                raise ValueError(f"Missing OUTCAR runs {cfg['missing_outcar_runs']}; elapsed time would be incomplete.")
            settings = {"kind": kind, "mapping": cfg.get("thermo_column_map"),
                        "timestep_fs": cfg.get("thermo_timestep_fs")}
            files = [(str(p.resolve()), p.stat().st_size, p.stat().st_mtime_ns) for p in paths]
            digest = hashlib.sha256(json.dumps({"version": 1, "settings": settings, "files": files}, default=str).encode()).hexdigest()[:16]
            cache = None if cache_dir is None else Path(cache_dir)/f"{cfg['metal']}_thermo_{digest}.npz"
            if use_cache and cache is not None and cache.is_file():
                with np.load(cache, allow_pickle=False) as saved:
                    raw = json.loads(str(saved["_metadata"]))
                    raw.update({k: saved[k].copy() for k in saved.files if k != "_metadata"})
            else:
                print(f"{cfg['label']}: reading thermodynamics from {kind}")
                raw = read_thermodynamics_csv(paths[0], cfg.get("thermo_column_map"), cfg.get("thermo_timestep_fs")) if kind == "csv" else read_outcar_thermodynamics(paths, cfg.get("thermo_timestep_fs"))
                if use_cache and cache is not None:
                    cache.parent.mkdir(parents=True, exist_ok=True)
                    arrays = {k: v for k, v in raw.items() if isinstance(v, np.ndarray)}
                    metadata = {k: v for k, v in raw.items() if k not in arrays}
                    np.savez_compressed(cache, **arrays, _metadata=np.array(json.dumps(metadata)))
            raw.update(label=cfg["label"], metal=cfg["metal"])
            return raw
        except (ValueError, FileNotFoundError) as exc:
            errors.append(f"{kind}: {exc}")
            if source == "auto" and kind == "csv" and paths and paths[0].is_file():
                warnings.warn(f"{cfg['label']}: thermo CSV could not be interpreted; trying OUTCARs. {exc}")
    raise ValueError(f"{cfg['label']}: no thermodynamics loaded. " + " | ".join(errors))


def discover_run_folder(folder):
    """Lightweight inventory: numeric folders, first POSCAR, and existing parts."""
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f"Metal folder does not exist: {folder}. Edit FOLDERS/DATA_ROOT.")
    parts = sorted((p for p in folder.iterdir() if p.is_dir() and p.name.isdecimal()),
                   key=lambda p: int(p.name))
    def first_existing(parent, names):
        return next((parent/name for name in names if (parent/name).is_file()), None)
    poscar = first_existing(folder, ("POSCAR", "POSCAR.gz", "poscar"))
    rows, outcars = [], []
    for p in parts:
        outcar = first_existing(p, ("OUTCAR", "OUTCAR.gz", "OUTCAR.bz2", "OUTCAR.xz"))
        pc = first_existing(p, ("POSCAR", "POSCAR.gz", "poscar"))
        if poscar is None and pc is not None:
            poscar = pc
        if outcar is not None:
            outcars.append(outcar)
        movie = folder/f"movie-file-{p.name}.xyz"
        temp = folder/f"file-{p.name}.csv"
        rows.append({"run_number": int(p.name), "folder": str(p),
                     "OUTCAR": str(outcar) if outcar else None,
                     "POSCAR": str(pc) if pc else None,
                     "CONTCAR": str(p/"CONTCAR") if (p/"CONTCAR").is_file() else None,
                     "part_movie_exists": movie.is_file(),
                     "part_movie_bytes": movie.stat().st_size if movie.is_file() else np.nan,
                     "part_temperature_csv_exists": temp.is_file()})
    if not parts:
        outcar = first_existing(folder, ("OUTCAR", "OUTCAR.gz", "OUTCAR.bz2", "OUTCAR.xz"))
        if outcar is not None:
            outcars = [outcar]
    return {"folder": folder, "poscar": poscar, "outcars": outcars,
            "parts": parts, "part_table": pd.DataFrame(rows),
            "missing_outcar_runs": [r["run_number"] for r in rows if r["OUTCAR"] is None]}


def restart_geometry_audit(layout, metal, tolerance_A=0.02):
    """Compare CONTCAR(n) to POSCAR(n+1); positions only, not velocities/state."""
    rows = []
    for previous, following in zip(layout["parts"][:-1], layout["parts"][1:]):
        row = {"previous_run": int(previous.name), "next_run": int(following.name),
               "nonconsecutive_run_numbers": int(following.name)-int(previous.name) != 1}
        final, initial = previous/"CONTCAR", following/"POSCAR"
        if not final.is_file() or not initial.is_file():
            rows.append({**row, "geometry_status": "CONTCAR/POSCAR unavailable"})
            continue
        a, b = read_poscar(final), read_poscar(initial)
        if not np.array_equal(a["symbols"], b["symbols"]) or not np.allclose(a["cell"], b["cell"], atol=1e-5):
            rows.append({**row, "geometry_status": "cell or atom order changed"})
            continue
        d = minimum_image(b["positions_A"]-a["positions_A"], a["cell"])
        distances = np.linalg.norm(d, axis=1)
        ids = np.flatnonzero(a["symbols"] == metal)
        rows.append({**row, "max_position_change_A": float(distances.max()),
                     "metal_position_change_A": float(distances[ids[0]]) if len(ids) == 1 else np.nan,
                     "geometry_status": "matches" if distances.max() <= tolerance_A else "position mismatch",
                     "velocities_checked": False})
    return pd.DataFrame(rows)


def load_run(cfg, workbook=None, cache_dir=None, use_cache=True):
    """Load one local run, optionally caching extracted arrays after streaming."""
    kind = cfg["kind"]
    poscar = cfg.get("poscar")
    structure = read_poscar(poscar) if poscar is not None and Path(poscar).is_file() else None
    neighbors = cfg.get("vacancy_neighbors")
    if isinstance(neighbors, str) and neighbors == "auto":
        proposals = [] if structure is None else suggest_b_vacancies(
            structure["positions_A"], structure["symbols"], structure["cell"])
        neighbors = proposals[0]["N_indices_zero_based"] if len(proposals) == 1 else None
        print(f"{cfg['label']}: {len(proposals)} candidate B-vacancy neighbor triplets; "
              f"selected {neighbors}. Inspect the proposed site in the trajectory plot.")
    if kind == "excel":
        paths = [Path(workbook)]
    elif kind == "csv":
        paths = [Path(cfg["csv"])]
    elif kind == "outcar":
        paths = [Path(p) for p in cfg["outcars"]]
        if not paths:
            raise FileNotFoundError(f"{cfg['label']}: no OUTCAR files were discovered.")
        if cfg.get("missing_outcar_runs") and not cfg.get("allow_missing_outcars", False):
            raise ValueError(f"Missing OUTCARs in runs {cfg['missing_outcar_runs']}. "
                             "Use the combined movie/CSV or resolve the missing files first.")
    elif kind == "xyz":
        paths = [Path(cfg["movie"])]
    else:
        raise ValueError("kind must be excel, csv, outcar, or xyz.")
    for p in paths:
        if not p.is_file():
            raise FileNotFoundError(f"{cfg['label']}: edit the settings for missing local file {p}")
    fingerprint_paths = paths + ([Path(poscar)] if structure is not None else [])
    fingerprint = {"version": 2, "settings": cfg,
                   "files": [(str(p.resolve()), p.stat().st_size, p.stat().st_mtime_ns) for p in fingerprint_paths]}
    key = hashlib.sha256(json.dumps(fingerprint, sort_keys=True, default=str).encode()).hexdigest()[:18]
    cache = None
    if use_cache and cache_dir is not None:
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
        cache = Path(cache_dir)/(key+"_raw.npz")
    if cache is not None and cache.exists():
        with np.load(cache, allow_pickle=False) as z:
            raw = json.loads(str(z["metadata_json"]))
            raw.update({k: z[k] for k in z.files if k != "metadata_json"})
        print(f"{cfg['label']}: loaded extracted coordinate cache")
        return raw
    print(f"{cfg['label']}: reading {kind} from local storage")
    if kind == "excel":
        raw = read_excel_trajectory(workbook, cfg["sheet"], cfg["metal"])
    elif kind == "csv":
        raw = read_csv_trajectory(cfg["csv"], cfg["metal"], cfg.get("csv_column_map"))
    elif kind == "outcar":
        raw = read_outcars(cfg["outcars"], cfg["metal"], cfg.get("metal_index"),
                           cfg.get("timestep_fs"), cfg.get("frame_interval_steps", 1), neighbors)
    elif kind == "xyz":
        raw = read_xyz_movie(cfg["movie"], cfg["metal"], structure,
                            cfg.get("metal_index"), cfg.get("frame_dt_fs", 0.5), neighbors,
                            cfg.get("xyz_time_unit"))
    else:
        raise ValueError("kind must be excel, csv, outcar, or xyz.")
    raw = attach_structure(raw, structure, neighbors)
    if cache is not None:
        arrays = {k: v for k, v in raw.items() if isinstance(v, np.ndarray)}
        metadata = {k: v for k, v in raw.items() if k not in arrays}
        np.savez_compressed(cache, metadata_json=json.dumps(metadata, default=str), **arrays)
    return raw


def read_outcars(paths, metal, metal_index=None, timestep_fs=None,
                 frame_interval_steps=1, vacancy_neighbors=None, progress_every=25000):
    """Stream POSITION/TOTAL-FORCE blocks and their ionic-step thermodynamics.

    paths must be in chronological order and have the same atom ordering/cell.
    Atom indices are zero-based. Repeated boundary frames are retained and flagged
    by the diagnostics, not silently deleted. Variable-cell MD is not supported.
    """
    paths = [Path(p).expanduser() for p in paths]
    times, positions, drifts, heights, temperatures, energies = [], [], [], [], [], []
    centers, normals, source_ids, ionic_steps = [], [], [], []
    cell0 = None
    symbols0 = None
    first_positions = None
    previous_full = None
    unwrapped_full = None
    substrate0 = None
    selected_index = None
    species_seen = []
    normal0 = None
    basis = None

    for source_id, path in enumerate(paths):
        if not path.is_file():
            raise FileNotFoundError(path)
        species, counts, cell = [], None, None
        dt_fs = timestep_fs
        current_step, record = None, {}
        previous_step_file = None
        emitted_file = 0
        with text_file(path) as handle:
            def emit(rec):
                nonlocal previous_full, unwrapped_full, first_positions, substrate0
                nonlocal cell0, symbols0, normal0, basis, selected_index
                nonlocal previous_step_file, emitted_file
                if "positions" not in rec:
                    return
                if cell is None or counts is None or len(species) < len(counts):
                    raise ValueError(f"{path}: could not identify cell, species, and atom counts.")
                if dt_fs is None or dt_fs <= 0:
                    raise ValueError(f"{path}: POTIM unavailable; set timestep_fs explicitly.")
                sym = np.repeat(species[:len(counts)], counts)
                full = rec["positions"]
                if first_positions is None:
                    cell0, symbols0 = np.array(cell), np.array(sym)
                    first_positions = full.copy()
                    idx = np.flatnonzero(sym == metal)
                    if metal_index is None:
                        if len(idx) != 1:
                            raise ValueError(f"Found {len(idx)} {metal} atoms; set metal_index explicitly.")
                        selected_index = int(idx[0])
                    else:
                        selected_index = int(metal_index)
                        if not 0 <= selected_index < len(sym) or sym[selected_index] != metal:
                            raise ValueError("metal_index does not point to the requested metal.")
                    substrate = np.isin(sym, ("B", "N"))
                    if not np.any(substrate):
                        raise ValueError("No B/N substrate atoms found.")
                    normal0 = np.cross(cell0[0], cell0[1])
                    normal0 /= np.linalg.norm(normal0)
                    e1 = cell0[0] / np.linalg.norm(cell0[0])
                    basis = np.vstack([e1, np.cross(normal0, e1), normal0])
                    unwrapped_full = full.copy()
                    substrate0 = unwrapped_full[substrate].mean(axis=0)
                else:
                    if not np.array_equal(sym, symbols0) or not np.allclose(cell, cell0, atol=1e-5):
                        raise ValueError("Atom order/cell changed. Analyze these runs separately.")
                    # For small MD increments, fractional image counters remove wrapping.
                    fstep = (full - previous_full) @ np.linalg.inv(cell0)
                    fstep[:, :2] -= np.rint(fstep[:, :2])
                    unwrapped_full += fstep @ cell0
                substrate = np.isin(sym, ("B", "N"))
                sheet_center = unwrapped_full[substrate].mean(axis=0)
                drift = sheet_center - substrate0
                height = np.dot(unwrapped_full[selected_index] - sheet_center, normal0)
                step = rec.get("ionic_step")
                if not times:
                    time = 0.0
                elif emitted_file and step is not None and previous_step_file is not None:
                    delta_steps = step - previous_step_file
                    if delta_steps <= 0:
                        raise ValueError("Ionic step reset inside one OUTCAR; split it at the restart.")
                    time = times[-1] + delta_steps * dt_fs / 1000
                else:
                    time = times[-1] + frame_interval_steps * dt_fs / 1000
                times.append(time)
                positions.append(full[selected_index].copy())
                drifts.append(drift)
                heights.append(height)
                temperatures.append(rec.get("temperature", np.nan))
                energies.append(rec.get("energy", np.nan))
                source_ids.append(source_id)
                ionic_steps.append(step if step is not None else np.nan)
                if vacancy_neighbors is not None:
                    ids = np.asarray(vacancy_neighbors, int)
                    if len(ids) != 3 or np.any(symbols0[ids] != "N"):
                        raise ValueError("vacancy_neighbors must contain three zero-based N indices.")
                    tri = full[ids].copy()
                    tri[1:] = tri[0] + minimum_image(tri[1:] - tri[0], cell0)
                    center = tri.mean(axis=0)
                    n = np.cross(tri[1] - tri[0], tri[2] - tri[0])
                    n /= np.linalg.norm(n)
                    if np.dot(n, normal0) < 0:
                        n = -n
                    centers.append(center)
                    normals.append(n)
                previous_full = full.copy()
                previous_step_file = step
                emitted_file += 1
                if progress_every and len(times) % progress_every == 0:
                    print(f"Read {len(times):,} frames ({time:.3f} ps)")

            for line in handle:
                match = re.search(r"VRHFIN\s*=\s*([A-Za-z]{1,2})\s*:", line)
                if match:
                    species.append(match.group(1))
                if "ions per type" in line:
                    counts = [int(v) for v in line.split("=", 1)[1].split()]
                match = re.search(r"POTIM\s*=\s*(" + FLOAT + ")", line)
                if match and timestep_fs is None:
                    dt_fs = number(match.group(1))
                match = re.search(r"IBRION\s*=\s*(-?\d+)", line)
                if match and int(match.group(1)) != 0:
                    raise ValueError(f"{path}: IBRION is not zero; this is not an MD OUTCAR.")
                if "direct lattice vectors" in line:
                    cell = np.array([[number(v) for v in next(handle).split()[:3]] for _ in range(3)])
                match = re.search(r"Iteration\s+(\d+)\s*\(\s*\d+\s*\)", line)
                if match:
                    step = int(match.group(1))
                    if current_step is not None and step != current_step:
                        emit(record)
                        record = {}
                    current_step = step
                    record["ionic_step"] = step
                match = re.search(r"free\s+energy\s+TOTEN\s*=\s*(" + FLOAT + ")", line)
                if match:
                    record["energy"] = number(match.group(1))
                match = re.search(r"temperature\s*=?\s*(" + FLOAT + r")\s*K", line, re.I)
                if match:
                    record["temperature"] = number(match.group(1))
                if "POSITION" in line and "TOTAL-FORCE" in line:
                    if counts is None:
                        raise ValueError("Cannot read positions without ions-per-type metadata.")
                    if "positions" in record:
                        emit(record)
                        record = {"ionic_step": current_step}
                    block = []
                    for coordinate_line in handle:
                        values = coordinate_line.split()
                        if not block and (not values or set(coordinate_line.strip()) == {"-"}):
                            continue
                        try:
                            row = [number(v) for v in values[:3]]
                        except ValueError:
                            break
                        if len(row) != 3:
                            break
                        block.append(row)
                        if len(block) == sum(counts):
                            break
                    if len(block) != sum(counts):
                        warnings.warn(f"Ignored incomplete final coordinate block in {path}")
                        break
                    record["positions"] = np.array(block)
            emit(record)
        if emitted_file == 0:
            raise ValueError(f"No complete POSITION blocks found in {path}.")
    if not times:
        raise ValueError("No trajectory frames read.")
    result = {"time_ps": np.array(times), "positions_A": np.array(positions),
              "substrate_drift_A": np.array(drifts), "height_A": np.array(heights),
              "temperature_K": np.array(temperatures), "energy_eV": np.array(energies),
              "cell": cell0, "basis": basis, "symbols": symbols0,
              "snapshot_A": first_positions, "metal_index": selected_index,
              "source_ids": np.array(source_ids), "ionic_steps": np.array(ionic_steps),
              "metal": metal, "source": [str(p) for p in paths]}
    if centers:
        result.update(vacancy_centers_A=np.array(centers), vacancy_normals=np.array(normals))
    return result


def suggest_b_vacancies(snapshot, symbols, cell, bond_cutoff_A=1.85):
    """Propose three undercoordinated N neighbors; inspect before accepting."""
    b = np.flatnonzero(np.asarray(symbols) == "B")
    n = np.flatnonzero(np.asarray(symbols) == "N")
    candidates = []
    for i in n:
        distance = np.linalg.norm(minimum_image(snapshot[b] - snapshot[i], cell), axis=1)
        if np.sum(distance < bond_cutoff_A) == 2:
            candidates.append(int(i))
    proposals, used = [], set()
    for i in candidates:
        near = [j for j in candidates if np.linalg.norm(minimum_image(snapshot[j] - snapshot[i], cell)) < 3.2]
        if len(near) == 3 and tuple(sorted(near)) not in used:
            ids = tuple(sorted(near))
            used.add(ids)
            tri = snapshot[list(ids)].copy()
            tri[1:] = tri[0] + minimum_image(tri[1:] - tri[0], cell)
            proposals.append({"N_indices_zero_based": ids, "center_A": tri.mean(axis=0).tolist()})
    return proposals


def prepare_trajectory(raw, cfg, jump_speed_A_ps=50.0):
    result = dict(raw)
    t = np.asarray(raw["time_ps"], float)
    p = np.asarray(raw["positions_A"], float)
    if len(t) < 3 or not np.all(np.isfinite(p)) or not np.all(np.isfinite(t)):
        raise ValueError("Need at least three finite trajectory frames.")
    if np.any(np.diff(t) <= 0):
        raise ValueError("Time is not strictly increasing; preserve/split the original restart files.")
    cell = cfg.get("cell_A")
    if cell is None:
        cell = raw.get("cell")
    if cell is not None:
        cell = np.asarray(cell, float)
        if cell.shape != (3, 3) or abs(np.linalg.det(cell)) < 1e-9:
            raise ValueError("cell_A must be three noncoplanar row vectors in Angstrom.")
    unwrapped = unwrap_positions(p, cell)
    drift = raw.get("substrate_drift_A", np.zeros_like(p))
    relative = unwrapped - drift
    basis = raw.get("basis", np.eye(3))
    projected = relative @ np.asarray(basis).T
    speed = np.linalg.norm(np.diff(relative, axis=0), axis=1) / np.diff(t)
    breaks = np.flatnonzero(speed > jump_speed_A_ps) + 1
    # Explicit source boundaries protect fits even when a restart gap looks small.
    if "source_ids" in raw and cfg.get("split_restart_boundaries", True):
        breaks = np.union1d(breaks, np.flatnonzero(np.diff(raw["source_ids"]) != 0) + 1)
    segment = np.zeros(len(t), int)
    segment[np.asarray(breaks, int)] = 1
    segment = np.cumsum(segment)
    vacancy = cfg.get("vacancy_cart_A")
    if vacancy is None and cfg.get("vacancy_frac") is not None:
        if cell is None:
            raise ValueError("A cell is required for vacancy_frac.")
        vacancy = np.asarray(cfg["vacancy_frac"], float) @ cell
    centers = raw.get("vacancy_centers_A")
    if centers is None and vacancy is not None:
        # A fixed site follows the measured substrate translation, if available.
        centers = np.asarray(vacancy, float)[None, :] + drift
    distance = height = local_height = None
    if centers is not None:
        delta = minimum_image(unwrapped - centers, cell)
        normal = np.asarray(basis)[2]
        height_along_normal = delta @ normal
        lateral = delta - height_along_normal[:, None] * normal
        distance = np.linalg.norm(lateral, axis=1)
        if raw.get("vacancy_normals") is not None:
            local_height = np.sum(delta * raw["vacancy_normals"], axis=1)
    height = raw.get("height_A")
    if local_height is not None:
        height = local_height
    elif height is None and cfg.get("sheet_plane_z_A") is not None:
        height = p[:, 2] - float(cfg["sheet_plane_z_A"])
    result.update(label=cfg["label"], condition=cfg.get("condition", "unassigned"),
                  replicate=cfg.get("replicate", "1"), cell=cell, projected_A=projected,
                  corrected_A=relative, segment_id=segment, break_indices=np.asarray(breaks, int),
                  jump_speed_A_ps=speed, distance_vacancy_A=distance, height_A=height,
                  drift_corrected=("substrate_drift_A" in raw))
    if centers is not None:
        result["vacancy_centers_A"] = centers
    if local_height is not None:
        result["height_reference"] = "instantaneous local three-N plane"
    return result


def trajectory_table(traj):
    t = traj["time_ps"]
    df = pd.DataFrame({"time_ps": t, "segment": traj["segment_id"]})
    for j, name in enumerate(("x", "y", "z")):
        df[name + "_raw_A"] = traj["positions_A"][:, j]
        df[name + "_corrected_A"] = traj["corrected_A"][:, j]
    for key, name in [("height_A", "height_A"), ("distance_vacancy_A", "r_vacancy_xy_A"),
                      ("temperature_K", "temperature_K"), ("energy_eV", "potential_energy_eV")]:
        if traj.get(key) is not None:
            df[name] = traj[key]
    return df


def inventory(trajectories):
    return pd.DataFrame([{
        "label": r["label"], "condition": r["condition"], "metal": r["metal"],
        "frames": len(r["time_ps"]), "duration_ps": float(np.ptp(r["time_ps"])),
        "dt_fs": 1000 * float(np.median(np.diff(r["time_ps"]))),
        "segments": int(r["segment_id"][-1] + 1), "cell_known": r["cell"] is not None,
        "substrate_drift_removed": r["drift_corrected"],
        "height_available": r.get("height_A") is not None,
        "height_reference": r.get("height_reference", "instantaneous B/N plane" if r["drift_corrected"] else "unavailable"),
        "vacancy_distance_available": r.get("distance_vacancy_A") is not None,
    } for r in trajectories])


def discontinuities(traj):
    rows = []
    for i in traj["break_indices"]:
        rows.append({"label": traj["label"], "frame_after": int(i),
                     "time_before_ps": traj["time_ps"][i-1], "time_after_ps": traj["time_ps"][i],
                     "step_A": np.linalg.norm(traj["corrected_A"][i] - traj["corrected_A"][i-1]),
                     "speed_A_ps": traj["jump_speed_A_ps"][i-1]})
    return pd.DataFrame(rows)


def choose_window(traj, time_window=(None, None), segment_id=None, stride=10):
    t = traj["time_ps"]
    low = t[0] if time_window[0] is None else time_window[0]
    high = t[-1] if time_window[1] is None else time_window[1]
    valid = (t >= low) & (t <= high)
    if segment_id is None:
        ids, counts = np.unique(traj["segment_id"][valid], return_counts=True)
        if len(ids) == 0:
            raise ValueError("No frames in the analysis window.")
        segment_id = int(ids[np.argmax(counts)])
    indices = np.flatnonzero(valid & (traj["segment_id"] == segment_id))[::int(stride)]
    if len(indices) < 20:
        raise ValueError("Too few frames in the selected continuous segment/window.")
    dt = np.diff(t[indices])
    if not np.allclose(dt, np.median(dt), rtol=1e-4, atol=1e-9):
        raise ValueError("MSD needs uniformly spaced frames; analyze a uniformly sampled section.")
    return indices


def msd_fft(positions):
    """Time-origin averaged component MSD, O(N log N), without crossing gaps."""
    p = np.asarray(positions, float)
    p = p - p.mean(axis=0)
    n = len(p)
    lag = np.arange(n)
    result = np.empty_like(p)
    for j in range(p.shape[1]):
        x = p[:, j]
        prefix = np.r_[0.0, np.cumsum(x*x)]
        ac = fftconvolve(x, x[::-1], mode="full")[n-1:]
        result[:, j] = (prefix[n-lag] + prefix[n] - prefix[lag] - 2*ac) / (n-lag)
    result[0] = 0.0
    return np.maximum(result, 0.0)


def msd_analysis(traj, time_window=(None, None), segment_id=None, stride=10, max_lag_ps=10.0):
    ids = choose_window(traj, time_window, segment_id, stride)
    p = traj["projected_A"][ids]
    t = traj["time_ps"][ids]
    dt = np.median(np.diff(t))
    # At most one quarter of the selected duration; long-lag samples are sparse.
    last = min(int(max_lag_ps/dt), (len(t)-1)//4)
    msd = msd_fft(p)[:last+1]
    tau = np.arange(last+1)*dt
    result = pd.DataFrame({"lag_ps": tau, "MSD_x_A2": msd[:, 0],
                           "MSD_y_A2": msd[:, 1], "MSD_z_A2": msd[:, 2],
                           "MSD_xy_A2": msd[:, 0]+msd[:, 1], "time_origins": len(t)-np.arange(last+1)})
    info = {"label": traj["label"], "analysis_start_ps": t[0], "analysis_end_ps": t[-1],
            "segment_id": int(traj["segment_id"][ids[0]]), "sample_dt_ps": dt,
            "selected_frames": len(t)}
    return result, info, ids


def fit_diffusion(msd, lag_window_ps, trajectory=None, selected_indices=None):
    """A candidate coefficient from a user-selected stationary diffusive window.

    R2 is descriptive. Correlated MSD points do not supply independent fit errors.
    The caller must assess stationarity and normal (alpha~1) diffusion.
    """
    low, high = lag_window_ps
    selected = (msd.lag_ps >= low) & (msd.lag_ps <= high) & (msd.lag_ps > 0)
    if selected.sum() < 5 or high > msd.lag_ps.max() + 1e-9:
        raise ValueError("Choose a fit interval within the computed MSD range, with >=5 points.")
    x = msd.loc[selected, "lag_ps"].to_numpy()
    y = msd.loc[selected, "MSD_xy_A2"].to_numpy()
    fit = linregress(x, y)
    exponent = linregress(np.log(x), np.log(np.maximum(y, 1e-30))).slope
    dx = linregress(x, msd.loc[selected, "MSD_x_A2"]).slope/2
    dy = linregress(x, msd.loc[selected, "MSD_y_A2"]).slope/2
    candidate = fit.slope / 4
    checks = []
    if candidate <= 0:
        checks.append("nonpositive slope")
    if not 0.8 <= exponent <= 1.2:
        checks.append("MSD exponent outside 0.8-1.2")
    if fit.rvalue**2 < 0.95:
        checks.append("R2 below 0.95")
    if trajectory is not None and trajectory["cell"] is None:
        checks.append("periodic cell not supplied")
    if trajectory is not None and not trajectory["drift_corrected"]:
        checks.append("substrate drift not available")
    return {"D_candidate_A2_ps": candidate, "D_candidate_m2_s": candidate*1e-8,
            "D_candidate_cm2_s": candidate*1e-4, "Dx_A2_ps": dx, "Dy_A2_ps": dy,
            "fit_intercept_A2": fit.intercept, "fit_slope_A2_ps": fit.slope,
            "R2": fit.rvalue**2, "alpha_fit": exponent,
            "fit_start_lag_ps": low, "fit_end_lag_ps": high,
            "assessment": "; ".join(checks) if checks else "candidate: verify stationarity and block consistency"}


def block_diffusion_estimates(traj, indices, lag_window_ps, blocks=4):
    """Descriptive variability across nonoverlapping time blocks, not a CI."""
    rows = []
    for j, ids in enumerate(np.array_split(indices, blocks)):
        if len(ids) < 20:
            continue
        time = traj["time_ps"][ids]
        dt = np.median(np.diff(time))
        last = min((len(ids)-1)//4, int(lag_window_ps[1]/dt)+1)
        if last*dt < lag_window_ps[1]:
            continue
        components = msd_fft(traj["projected_A"][ids])[:last+1]
        frame = pd.DataFrame({"lag_ps": np.arange(last+1)*dt,
                              "MSD_x_A2": components[:, 0], "MSD_y_A2": components[:, 1],
                              "MSD_xy_A2": components[:, 0]+components[:, 1]})
        f = fit_diffusion(frame, lag_window_ps, traj)
        rows.append({"label": traj["label"], "block": j+1, "start_ps": time[0], "end_ps": time[-1], **f})
    return pd.DataFrame(rows)


def characteristic_diffusion_time(D_A2_ps, length_A):
    """2D normal-diffusion timescale L^2/(4D); not a vacancy-capture rate."""
    if D_A2_ps <= 0 or length_A <= 0:
        return np.nan
    return length_A**2 / (4*D_A2_ps)


def persistent_runs(mask, time, segments, dwell_ps):
    mask = np.asarray(mask, bool)
    boundaries = np.r_[0, np.flatnonzero((mask[1:] != mask[:-1]) | (segments[1:] != segments[:-1]))+1, len(mask)]
    return [(int(a), int(b)) for a, b in zip(boundaries[:-1], boundaries[1:])
            if mask[a] and time[b-1] - time[a] >= dwell_ps]


def capture_metrics(traj, radius_A=1.5, dwell_ps=0.5):
    distance = traj.get("distance_vacancy_A")
    if distance is None:
        return {"capture_status": "vacancy position not supplied"}
    t, segments = traj["time_ps"], traj["segment_id"]
    visits = persistent_runs(distance <= radius_A, t, segments, dwell_ps)
    if not visits:
        gapped = len(traj["break_indices"]) > 0
        return {"capture_status": "no entry in observed pieces; trajectory has gaps" if gapped else "no persistent entry observed",
                "capture_observed": False,
                "capture_censor_time_ps": np.nan if gapped else float(t[-1]-t[0]), "capture_radius_A": radius_A}
    first, end = visits[0]
    if first == 0:
        return {"capture_status": "already inside at first frame; entry is left-censored",
                "capture_observed": False, "first_persistent_entry_ps": np.nan,
                "capture_radius_A": radius_A}
    gap_before = bool(np.any(traj["break_indices"] <= first))
    reaches_boundary = end == len(t) or (end < len(t) and segments[end] != segments[end-1])
    inside_durations = [float(t[b-1]-t[a]) for a, b in visits]
    return {"capture_status": "entry after a data gap; timing uncertain" if gap_before else "persistent entry observed",
            "capture_observed": not gap_before,
            "first_persistent_entry_ps": float(t[first]-t[0]),
            "entry_confirmed_ps": float(t[first]-t[0]+dwell_ps),
            "capture_radius_A": radius_A, "capture_dwell_ps": dwell_ps,
            "first_residence_observed_ps": float(t[end-1]-t[first]),
            "first_residence_right_censored": reaches_boundary,
            "persistent_visits": len(visits), "inside_time_fraction": float(np.mean(distance <= radius_A)),
            "observed_visit_durations_ps": inside_durations}


def crossing_events(traj, deadband_A=0.25, dwell_ps=0.1):
    height = traj.get("height_A")
    if height is None:
        return pd.DataFrame()
    t, segments = traj["time_ps"], traj["segment_id"]
    episodes = []
    for side, mask in [(1, height >= deadband_A), (-1, height <= -deadband_A)]:
        episodes += [(a, b, side) for a, b in persistent_runs(mask, t, segments, dwell_ps)]
    episodes.sort()
    previous = None
    events = []
    for a, b, side in episodes:
        if previous is not None and previous[2] != side and segments[a] == segments[previous[0]]:
            events.append({"label": traj["label"], "time_ps": float(t[a]),
                           "direction": "above to below" if side < 0 else "below to above",
                           "r_vacancy_xy_A": float(traj["distance_vacancy_A"][a]) if traj.get("distance_vacancy_A") is not None else np.nan,
                           "deadband_A": deadband_A, "dwell_ps": dwell_ps})
        previous = (a, b, side)
    return pd.DataFrame(events)


def exponential_capture_tau(event_times_ps, event_observed):
    """Right-censored exponential MLE for independently prepared equivalent runs.

    Do not combine different metals or starting-distance groups into one rate.
    """
    t, e = np.asarray(event_times_ps, float), np.asarray(event_observed, bool)
    if len(t) < 3:
        raise ValueError("Supply at least three independent comparable trajectories; more are preferable.")
    if np.any(t < 0) or len(t) != len(e) or not np.all(np.isfinite(t)):
        raise ValueError("Supply finite nonnegative durations and matching event flags.")
    events = int(e.sum())
    return {"replicates": len(t), "capture_events": events,
            "tau_exponential_ps": float(t.sum()/events) if events else np.nan,
            "rate_per_ps": float(events/t.sum()) if t.sum() else np.nan}


def periodic_sites_in_view(points, cell, basis, xlim, ylim):
    """Project/tile real reference coordinates into an unwrapped path's view."""
    xy = np.asarray(points) @ np.asarray(basis).T
    if cell is None or not len(xy):
        return xy[:, :2]
    lattice = (np.asarray(cell)[:2] @ np.asarray(basis).T)[:, :2]
    inverse = np.linalg.inv(lattice)
    corners = np.array(list(product(xlim, ylim))) @ inverse
    fractional_sites = xy[:, :2] @ inverse
    lo = np.floor(corners.min(axis=0)-fractional_sites.max(axis=0)).astype(int)-1
    hi = np.ceil(corners.max(axis=0)-fractional_sites.min(axis=0)).astype(int)+1
    if np.prod(hi-lo+1) > 10000:
        raise ValueError("Path spans too many periodic images for a lattice background; disable SHOW_LATTICE.")
    tiles = []
    for i, j in product(range(lo[0], hi[0]+1), range(lo[1], hi[1]+1)):
        tile = xy[:, :2] + np.array([i, j]) @ lattice
        inside = ((tile[:, 0] >= xlim[0]) & (tile[:, 0] <= xlim[1])
                  & (tile[:, 1] >= ylim[0]) & (tile[:, 1] <= ylim[1]))
        tiles.append(tile[inside])
    return np.concatenate(tiles) if tiles else np.empty((0, 2))


def draw_migration_xy(trajectories, show_lattice=True, lattice_colors=None,
                      lattice_alpha=0.40, lattice_dot_size=25, cmap="viridis",
                      padding_A=2.5, line_width=None, max_plot_points=12000,
                      time_limits_ps=None, show_colorbar=False):
    colors = lattice_colors or {"B": "#249B59", "N": "#3659C9"}
    count = len(trajectories)
    fig, axes = plt.subplots(1, count, figsize=(7.6*count, 8.5), squeeze=False)
    bounds = time_limits_ps if time_limits_ps is not None else (0, max(r["time_ps"][-1] for r in trajectories))
    norm = Normalize(*bounds)
    width = plot_line_width() if line_width is None else float(line_width)
    for ax, r in zip(axes[0], trajectories):
        p, t = r["projected_A"], r["time_ps"]
        center = (p[:, :2].min(axis=0)+p[:, :2].max(axis=0))/2
        span = max(float(np.ptp(p[:, 0])), float(np.ptp(p[:, 1])))+2*padding_A
        xlim, ylim = (center[0]-span/2, center[0]+span/2), (center[1]-span/2, center[1]+span/2)
        basis = r.get("basis", np.eye(3))
        if show_lattice and r.get("snapshot_A") is not None:
            symbols = np.asarray(r["symbols"])
            for element in ("B", "N"):
                points = r["snapshot_A"][symbols == element]
                if len(points):
                    xy = periodic_sites_in_view(points, r["cell"], basis, xlim, ylim)
                    ax.scatter(xy[:, 0], xy[:, 1], s=lattice_dot_size, c=colors[element],
                               alpha=lattice_alpha, edgecolors="none", zorder=0,
                               label=element, rasterized=True)
        for sid in np.unique(r["segment_id"]):
            ids = np.flatnonzero(r["segment_id"] == sid)
            stride = max(1, int(np.ceil(len(ids)/max_plot_points)))
            ids = np.unique(np.r_[ids[::stride], ids[-1]])
            if len(ids) > 1:
                xy = p[ids, :2]
                lines = LineCollection(np.stack([xy[:-1], xy[1:]], axis=1),
                                       cmap=cmap, norm=norm, linewidth=width, zorder=3)
                lines.set_array(t[ids[:-1]])
                ax.add_collection(lines)
        ax.scatter(*p[0, :2], s=180, marker="o", facecolors="white", edgecolors="black",
                   linewidth=width, zorder=6, label="Start")
        ax.scatter(*p[-1, :2], s=130, marker="s", c=COLORS.get(r["metal"], "black"),
                   edgecolors="black", linewidth=width, zorder=7, label="End")
        if r.get("vacancy_centers_A") is not None:
            xy = periodic_sites_in_view(np.asarray(r["vacancy_centers_A"])[0:1], r["cell"], basis, xlim, ylim)
            ax.scatter(xy[:, 0], xy[:, 1], s=160, marker="x", c="#882255", linewidth=width,
                       zorder=5, label=r"$V_\mathrm{B}$")
        ax.set(xlim=xlim, ylim=ylim, xlabel=r"$x$ (Å)", ylabel=r"$y$ (Å)", title=r["label"])
        ax.set_aspect("equal", adjustable="box")
        ax.spines[["top", "right"]].set_visible(False)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.54, 1.0),
               ncol=3 if count == 1 else len(labels), frameon=False,
               handletextpad=0.3, columnspacing=1.0)
    fig.subplots_adjust(left=0.18 if count == 1 else 0.075, right=0.97,
                        bottom=0.15 if show_colorbar else 0.10, top=0.81, wspace=0.40)
    if show_colorbar:
        from matplotlib.cm import ScalarMappable
        cax = fig.add_axes([0.28, 0.025, 0.5, 0.025])
        fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal").set_label("Time (ps)")
    return fig



def draw_height_history(trajectories, max_plot_points=10000):
    fig, axes = plt.subplots(1, len(trajectories), figsize=(7*len(trajectories), 5.7), squeeze=False)
    for ax, r in zip(axes[0], trajectories):
        height = r.get("height_A")
        y = height if height is not None else r["projected_A"][:, 2]-r["projected_A"][0, 2]
        for sid in np.unique(r["segment_id"]):
            ids = np.flatnonzero(r["segment_id"] == sid)
            ids = ids[::max(1, int(np.ceil(len(ids)/max_plot_points)))]
            ax.plot(r["time_ps"][ids], y[ids], color=COLORS.get(r["metal"]), lw=plot_line_width())
        if height is not None:
            ax.axhline(0, color="#999999", ls="--", lw=plot_line_width())
        ax.set(xlabel="Time (ps)", ylabel="Height (Å)" if height is not None else r"$z-z(0)$ (Å)", title=r["label"])
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(pad=1.2)
    return fig



def draw_coordinate_history(trajectories, max_plot_points=10000):
    fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
    for r in trajectories:
        p = r["projected_A"]-r["projected_A"][0]
        for sid in np.unique(r["segment_id"]):
            ids = np.flatnonzero(r["segment_id"] == sid)
            ids = ids[::max(1, int(np.ceil(len(ids)/max_plot_points)))]
            for k, ax in enumerate(axes):
                ax.plot(r["time_ps"][ids], p[ids, k], lw=plot_line_width(), c=COLORS.get(r["metal"]),
                        label=r["label"] if sid == r["segment_id"][0] else None)
    for ax, coordinate in zip(axes, ("x", "y", "z")):
        ax.set_ylabel(f"${coordinate}-{coordinate}(0)$ (Å)")
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].legend(frameon=False, ncol=len(trajectories), fontsize=plt.rcParams["axes.titlesize"])
    axes[-1].set_xlabel("Time (ps)")
    fig.tight_layout(pad=1.2)
    return fig



def draw_trajectory_overview(trajectories, font_scale=1.4, max_plot_points=6000):
    fig, axes = plt.subplots(2, len(trajectories), figsize=(4.5*len(trajectories), 7.2), squeeze=False)
    norm = Normalize(0, max(r["time_ps"][-1] for r in trajectories))
    artist = None
    for col, r in enumerate(trajectories):
        ax, az = axes[0, col], axes[1, col]
        t, p = r["time_ps"], r["projected_A"]
        if r.get("snapshot_A") is not None:
            fixed = r["snapshot_A"] @ np.asarray(r.get("basis", np.eye(3))).T
            for element, color in [("B", "#99BC90"), ("N", "#AEA0CB")]:
                ids = np.asarray(r["symbols"]) == element
                ax.scatter(fixed[ids, 0], fixed[ids, 1], s=13, c=color, alpha=0.45, zorder=0)
        for sid in np.unique(r["segment_id"]):
            ids = np.flatnonzero(r["segment_id"] == sid)
            ids = ids[::max(1, int(np.ceil(len(ids)/max_plot_points)))]
            if len(ids) > 1:
                xy = p[ids, :2]
                pieces = np.stack([xy[:-1], xy[1:]], axis=1)
                artist = LineCollection(pieces, cmap="viridis", norm=norm, linewidth=plot_line_width())
                artist.set_array(t[ids[:-1]])
                ax.add_collection(artist)
            az.plot(t[ids], (r["height_A"] if r.get("height_A") is not None else p[:, 2]-p[0, 2])[ids],
                    c=COLORS.get(r["metal"], "#444444"), lw=plot_line_width())
        ax.scatter(p[0, 0], p[0, 1], s=65, marker="o", facecolors="none", edgecolors="black", zorder=5)
        ax.scatter(p[-1, 0], p[-1, 1], s=40, marker="s", c=COLORS.get(r["metal"], "black"), zorder=5)
        if r.get("vacancy_centers_A") is not None:
            center = r["vacancy_centers_A"][0]
            # Plot the vacancy image nearest the region the corrected atom explores.
            target = r["corrected_A"].mean(axis=0)
            center = target + minimum_image(center-target, r["cell"])
            xy_center = center @ np.asarray(r.get("basis", np.eye(3))).T
            ax.scatter(xy_center[0], xy_center[1], marker="x", s=70, color="#882255", lw=plot_line_width(), zorder=7)
        ax.autoscale_view()
        ax.set_aspect("equal", adjustable="box")
        # Restrict a lattice background to the actual region traversed by the atom.
        if r.get("snapshot_A") is not None:
            ax.set_xlim(p[:, 0].min()-2, p[:, 0].max()+2)
            ax.set_ylim(p[:, 1].min()-2, p[:, 1].max()+2)
        ax.set_title(r["label"])
        ax.set_xlabel(r"$x$ (Å)")
        ax.set_ylabel(r"$y$ (Å)")
        az.set_xlabel("Time (ps)")
        az.set_ylabel("Height (Å)" if r.get("height_A") is not None else r"$z-z(0)$ (Å)")
        if r.get("height_A") is not None:
            az.axhline(0, color="#888888", ls="--", lw=plot_line_width())
        for a in (ax, az):
            a.spines[["top", "right"]].set_visible(False)
    fig.subplots_adjust(left=0.07, right=0.90, bottom=0.10, top=0.92, hspace=0.35, wspace=0.35)
    if artist is not None:
        bar = fig.colorbar(artist, cax=fig.add_axes([0.925, 0.57, 0.012, 0.30]))
        bar.set_label("Time (ps)")
    return fig


def draw_msd(results, fits=None):
    fig, axes = plt.subplots(1, 2, figsize=(14.8, 6.2))
    for label, bundle in results.items():
        msd, metal = bundle[0], bundle[3]
        color = COLORS.get(metal)
        axes[0].plot(msd.lag_ps, msd.MSD_xy_A2, label=label, color=color, lw=plot_line_width())
        select = (msd.lag_ps > 0) & (msd.MSD_xy_A2 > 0)
        axes[1].loglog(msd.loc[select, "lag_ps"], msd.loc[select, "MSD_xy_A2"],
                       label=label, color=color, lw=plot_line_width())
        if fits and label in fits:
            f = fits[label]
            x = np.array([f["fit_start_lag_ps"], f["fit_end_lag_ps"]])
            axes[0].plot(x, f["fit_intercept_A2"]+x*f["fit_slope_A2_ps"],
                         "--", color=color, lw=plot_line_width())
    for ax in axes:
        ax.set_xlabel("Lag time (ps)")
        ax.set_ylabel(r"In-plane MSD (Å$^2$)")
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].legend(frameon=False, fontsize=plt.rcParams["axes.titlesize"])
    fig.tight_layout(pad=1.3, w_pad=2)
    return fig



def draw_vacancy_distances(trajectories, radius_A=1.5):
    usable = [r for r in trajectories if r.get("distance_vacancy_A") is not None]
    if not usable:
        return None
    fig, axes = plt.subplots(len(usable), 1, figsize=(12, 4.8*len(usable)), squeeze=False, sharex=True)
    for ax, r in zip(axes[:, 0], usable):
        for sid in np.unique(r["segment_id"]):
            ids = np.flatnonzero(r["segment_id"] == sid)
            ax.plot(r["time_ps"][ids], r["distance_vacancy_A"][ids], lw=plot_line_width(),
                    c=COLORS.get(r["metal"], "black"))
        ax.axhline(radius_A, c="#882255", ls="--", lw=plot_line_width())
        ax.set_ylabel(r"$r_{V_B,xy}$ (Å)")
        ax.set_title(r["label"])
        ax.spines[["top", "right"]].set_visible(False)
    axes[-1, 0].set_xlabel("Time (ps)")
    fig.tight_layout(pad=1.2)
    return fig



def draw_thermodynamics(trajectories, energy_reference="initial", reference_temperature_K=300,
                        max_plot_points=10000, temperature_ylim=None, energy_ylims=None):
    usable = [r for r in trajectories if np.isfinite(r.get("temperature_K", [np.nan])).any()
              and np.isfinite(r.get("energy_eV", [np.nan])).any()]
    if not usable:
        return None
    if energy_reference not in ("initial", "absolute"):
        raise ValueError("THERMO_ENERGY_REFERENCE must be initial or absolute.")
    tlim = temperature_ylim if temperature_ylim is not None else globals().get("TEMPERATURE_YLIM", (200, 500))
    elims = energy_ylims if energy_ylims is not None else globals().get("ENERGY_YLIMS", {"Au": (-1, 1), "Ag": (-3, 1), "Cu": (-1, 1)})
    fig, axes = plt.subplots(2, len(usable), figsize=(6.8*len(usable), 9.0), squeeze=False, sharex="col")
    for column, r in enumerate(usable):
        t, energy = r["time_ps"], np.asarray(r["energy_eV"], float).copy()
        if energy_reference == "initial":
            energy -= energy[np.flatnonzero(np.isfinite(energy))[0]]
        source_ids = r.get("source_ids", np.zeros(len(t), int))
        for sid in np.unique(source_ids):
            ids = np.flatnonzero(source_ids == sid)
            ids = ids[::max(1, int(np.ceil(len(ids)/max_plot_points)))]
            axes[0, column].plot(t[ids], r["temperature_K"][ids], lw=plot_line_width(), c=COLORS.get(r["metal"]))
            axes[1, column].plot(t[ids], energy[ids], lw=plot_line_width(), c=COLORS.get(r["metal"]))
        if reference_temperature_K is not None:
            axes[0, column].axhline(reference_temperature_K, c="#999999", ls="--", lw=plot_line_width())
        axes[0, column].set(title=r["label"], ylabel="Temperature (K)", ylim=tlim)
        axes[0, column].set_yticks([200, 300, 400, 500] if tuple(tlim) == (200, 500) else np.linspace(*tlim, 4))
        axes[1, column].set(ylabel=r"$E-E_0$ (eV)" if energy_reference == "initial" else "Energy (eV)", xlabel="Time (ps)")
        if energy_reference == "initial" and r["metal"] in elims:
            axes[1, column].set_ylim(*elims[r["metal"]])
            low, high = elims[r["metal"]]
            axes[1, column].set_yticks(np.arange(low, high+0.01, 1))
        for ax in axes[:, column]:
            ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(pad=1.3, w_pad=2, h_pad=1.2)
    return fig



def make_xy_animation(traj, frames=300, tail_points=1000):
    """Return a notebook-ready animation; never connect across flagged gaps."""
    from matplotlib.animation import FuncAnimation
    t, p, seg = traj["time_ps"], traj["projected_A"], traj["segment_id"]
    ids = np.unique(np.linspace(0, len(t)-1, min(frames, len(t))).astype(int))
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(p[:, 0].min()-0.5, p[:, 0].max()+0.5)
    ax.set_ylim(p[:, 1].min()-0.5, p[:, 1].max()+0.5)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(r"$x$ (Å)")
    ax.set_ylabel(r"$y$ (Å)")
    line, = ax.plot([], [], lw=plot_line_width(), c=COLORS.get(traj["metal"], "black"))
    dot, = ax.plot([], [], "o", c=COLORS.get(traj["metal"], "black"))
    clock = ax.text(0.03, 0.95, "", transform=ax.transAxes)
    def update(i):
        lo = max(0, i-tail_points)
        take = np.arange(lo, i+1)
        take = take[seg[take] == seg[i]]
        line.set_data(p[take, 0], p[take, 1])
        dot.set_data([p[i, 0]], [p[i, 1]])
        clock.set_text(f"{traj['label']}   {t[i]:.2f} ps")
        return line, dot, clock
    return FuncAnimation(fig, update, frames=ids, interval=40, blit=True)

def plot_line_width():
    return float(globals().get("LINE_WIDTH", 4.0))



def apply_plot_style():
    sizes = globals().get("FONT_SIZES", {"base": 20, "axes": 26, "ticks": 20, "title": 26, "legend": 20})
    scale = float(globals().get("FONT_SCALE", 1.0))
    width = plot_line_width()
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": scale*sizes["base"],
        "axes.labelsize": scale*sizes["axes"], "axes.titlesize": scale*sizes["title"],
        "xtick.labelsize": scale*sizes["ticks"], "ytick.labelsize": scale*sizes["ticks"],
        "legend.fontsize": scale*sizes["legend"], "axes.linewidth": width,
        "lines.linewidth": width, "patch.linewidth": width,
        "xtick.major.width": width, "ytick.major.width": width,
        "xtick.minor.width": width, "ytick.minor.width": width,
        "xtick.major.size": 8, "ytick.major.size": 8,
        "xtick.minor.size": 4, "ytick.minor.size": 4,
        "svg.fonttype": "none", "pdf.fonttype": 42,
    })



def load_saved_raw(results_dir, metal, explicit_cache=None):
    """Use the existing full-geometry cache for styling, without reading a movie."""
    paths = [Path(explicit_cache)] if explicit_cache else sorted((Path(results_dir)/"cache").glob("*_raw.npz"))
    matches = []
    for path in paths:
        with np.load(path, allow_pickle=False) as data:
            if "metadata_json" not in data:
                continue
            metadata = json.loads(str(data["metadata_json"]))
            if metadata.get("metal") == metal:
                matches.append(path)
    if len(matches) != 1:
        raise ValueError(f"{metal}: found {len(matches)} matching saved raw caches. "
                         "Set REPLOT_RAW_CACHE[metal] to the intended cache, or REPLOT_ONLY=False.")
    with np.load(matches[0], allow_pickle=False) as data:
        raw = json.loads(str(data["metadata_json"]))
        raw.update({key: data[key].copy() for key in data.files if key != "metadata_json"})
    print(f"{metal}: using saved coordinate arrays ({len(raw['time_ps']):,} frames)")
    return raw



def load_saved_thermo(results_dir, prefix, metal):
    path = Path(results_dir)/f"{prefix}_temperature_energy_data.csv"
    data = pd.read_csv(path)
    quantity = "unspecified energy from saved CSV"
    reports = list(Path(results_dir).glob("*_thermodynamic_sources.csv"))
    if len(reports) == 1:
        report = pd.read_csv(reports[0])
        row = report[report.metal == metal]
        if len(row) == 1:
            quantity = str(row.iloc[0].energy_quantity)
    return {"time_ps": data.time_ps.to_numpy(float),
            "temperature_K": data.temperature_K.to_numpy(float),
            "energy_eV": data.energy_eV.to_numpy(float),
            "source_ids": data.source_segment.to_numpy(int) if "source_segment" in data else np.zeros(len(data), int),
            "label": metal, "metal": metal, "source": str(path), "energy_quantity": quantity}



def draw_migration_timebar(time_limits_ps, cmap="viridis"):
    from matplotlib.cm import ScalarMappable
    from matplotlib.ticker import MaxNLocator
    fig = plt.figure(figsize=(9.5, 2.2))
    cax = fig.add_axes([0.09, 0.52, 0.84, 0.23])
    bar = fig.colorbar(ScalarMappable(norm=Normalize(*time_limits_ps), cmap=cmap),
                       cax=cax, orientation="horizontal")
    bar.locator = MaxNLocator(nbins=5)
    bar.update_ticks()
    bar.set_label("Time (ps)", labelpad=10)
    bar.outline.set_linewidth(plot_line_width())
    return fig

In [ ]:
# These helpers give every output a descriptive, consistent name.
from io import BytesIO
OUTPUT_FILES = []

def prefix_for(label):
    return OUTPUT_PREFIXES.get(label, re.sub(r"[^A-Za-z0-9_.-]", "_", label))

def save_table(frame, name):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUTPUT_DIR / f"{name}.csv"
    frame.to_csv(path, index=False)
    OUTPUT_FILES.append(path)
    return path

def finish_figure(fig, name, show=None):
    if fig is None:
        return
    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        for extension in FIGURE_FORMATS:
            buffer = BytesIO()
            fig.savefig(buffer, format=extension, dpi=FIGURE_DPI,
                        bbox_inches="tight", facecolor="white")
            path = OUTPUT_DIR / f"{name}.{extension}"
            path.write_bytes(buffer.getvalue())
            OUTPUT_FILES.append(path)
    if SHOW_FIGURES and (show is None or show):
        plt.show()
    plt.close(fig)

## Check the folder layout first

Discovery uses `6nn/au-100ps`, `7nn/ag-100ps`, and `7nn/cu-100ps`. It locates `movie-combined.xyz`, the configured coordinate/thermodynamic CSVs, and `1/POSCAR` (or the earliest available POSCAR). For OUTCAR input, `1/OUTCAR`, `2/OUTCAR`, … are gathered in **numeric** order, including compressed OUTCARs. Other metal directories are excluded.

The part-file table records missing per-run movies/temperature CSVs. A missing temperature CSV does not establish that a coordinate segment is absent; review your combining-script log.

The restart audit compares positions and atom order in CONTCAR(n) and POSCAR(n+1). It does **not** verify transferred velocities or thermostat state. A continuous restart also requires appropriate continuation of those quantities. If a numbered folder was started independently, analyze it as a separate replica. Concatenation alone does not establish continuity.

For duration, count **ionic MD steps**, not SCF electronic iterations. At POTIM=0.5 fs, 8000 ionic steps correspond to 4 ps; 8000 electronic iterations do not determine a duration. XYZ spacing also depends on how often frames were saved/exported.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if REPLOT_ONLY:
    candidates = list(PREVIOUS_RESULTS_DIR.glob("*_analysis_settings.json"))
    if len(candidates) != 1:
        raise ValueError("Select the folder containing one saved *_analysis_settings.json in PREVIOUS_RESULTS_DIR.")
    previous_settings = json.loads(candidates[0].read_text())
    saved_runs = {r["metal"]: r for r in previous_settings["runs"]}
    RUNS = []
    for metal in ACTIVE_METALS:
        cfg = dict(saved_runs[metal])
        folder = DATA_ROOT / FOLDERS[metal]
        cfg.update(label=metal, metal=metal, output_prefix=OUTPUT_PREFIXES[metal], folder=folder,
                   thermo_csv=THERMO_CSV_PATHS.get(metal, folder/THERMO_CSV_NAMES[metal]), thermo_column_map=THERMO_COLUMN_MAP,
                   thermo_timestep_fs=THERMO_TIMESTEP_FS)
        RUNS.append(cfg)
    display(pd.DataFrame([{"metal": r["metal"], "mode": "reuse saved coordinate arrays",
                            "previous_results": str(PREVIOUS_RESULTS_DIR)} for r in RUNS]))
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    RUNS, layout_rows, restart_tables = [], [], []
    for metal in ACTIVE_METALS:
        folder = DATA_ROOT / FOLDERS[metal]
        layout = discover_run_folder(folder)
        movie, coordinate_csv = folder / MOVIE_NAMES[metal], folder / CSV_NAMES[metal]
        layout_rows.append({"metal": metal, "folder": str(folder),
                            "numbered_runs": len(layout["parts"]),
                            "POSCAR": str(layout["poscar"]), "OUTCAR_files": len(layout["outcars"]),
                            "missing_OUTCAR_runs": layout["missing_outcar_runs"],
                            "combined_XYZ_exists": movie.is_file(), "trajectory_CSV_exists": coordinate_csv.is_file()})
        save_table(layout["part_table"], f"{prefix_for(metal)}_part_file_inventory")
        if SHOW_PART_FILE_TABLES:
            display(layout["part_table"])
        if CHECK_RESTART_GEOMETRY:
            audit = restart_geometry_audit(layout, metal, RESTART_GEOMETRY_TOL_A)
            if not audit.empty:
                audit.insert(0, "metal", metal)
                restart_tables.append(audit)
        RUNS.append({
            "label": metal, "metal": metal, "condition": "VB", "replicate": "1",
            "output_prefix": OUTPUT_PREFIXES[metal], "folder": folder,
            "kind": SOURCE, "sheet": f"{metal}_trajectory", "poscar": layout["poscar"],
            "outcars": layout["outcars"], "missing_outcar_runs": layout["missing_outcar_runs"],
            "allow_missing_outcars": False,
            "movie": movie, "csv": coordinate_csv, "csv_column_map": CSV_COLUMN_MAP,
            "thermo_csv": THERMO_CSV_PATHS.get(metal, folder / THERMO_CSV_NAMES[metal]),
            "thermo_column_map": THERMO_COLUMN_MAP, "thermo_timestep_fs": THERMO_TIMESTEP_FS,
            "frame_dt_fs": XYZ_FRAME_DT_FS, "xyz_time_unit": None,
            "metal_index": None, "timestep_fs": None, "frame_interval_steps": 1,
            "split_restart_boundaries": SPLIT_RESTART_BOUNDARIES,
            "vacancy_neighbors": VACANCY_NEIGHBORS[metal],
            "vacancy_cart_A": None, "vacancy_frac": None, "cell_A": None, "sheet_plane_z_A": None,
        })
    display(pd.DataFrame(layout_rows))
    if restart_tables:
        restart_table = pd.concat(restart_tables, ignore_index=True)
        display(restart_table)
        save_table(restart_table, f"{COMPARISON_PREFIX}_restart_geometry_audit")

    # Optional header checks before loading a large movie:
    # display(pd.read_csv(RUNS[0]["csv"], nrows=3))
    # display(pd.read_csv(DATA_ROOT/FOLDERS["Au"]/"combined-temp-energy.csv", nrows=3))
    # A CSV with a generic Time header needs a unit mapping in THERMO_COLUMN_MAP.
    # Set THERMO_SOURCE="outcar" to read temperature/TOTEN directly, even with SOURCE="xyz".

# Fail before plotting if a requested fresh CSV is missing. Do not reuse stale exports.
if REFRESH_THERMODYNAMICS and THERMO_SOURCE == "csv":
    for cfg in RUNS:
        path = Path(cfg["thermo_csv"])
        if not path.is_file():
            raise FileNotFoundError(f"{cfg['metal']}: requested temperature/energy CSV does not exist: {path}")
        if cfg["metal"] == "Au":
            gold_csv_check = pd.read_csv(path)
            print(f"Au temperature/energy CSV: {path.resolve()}")
            print(f"Rows: {len(gold_csv_check):,}; columns: {list(gold_csv_check.columns)}")
            time_columns = [c for c in ("step", "time_fs", "time_ps") if c in gold_csv_check]
            display(gold_csv_check[time_columns].iloc[[0, -1]] if time_columns else gold_csv_check.iloc[[0, -1]])

## Reuse saved arrays and apply the figure style

`PREVIOUS_RESULTS_DIR` points to the completed run. Each metal is selected by metadata inside its raw NPZ cache, not by a guessed filename. More than one matching cache requires an explicit `REPLOT_RAW_CACHE` choice. Coordinate, MSD, and capture definitions are unchanged. Only plot style and the independent temperature/energy refresh are revised.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
apply_plot_style()
trajectories = []
for cfg in RUNS:
    raw = (load_saved_raw(PREVIOUS_RESULTS_DIR, cfg["metal"], REPLOT_RAW_CACHE.get(cfg["metal"]))
           if REPLOT_ONLY else load_run(cfg, workbook=WORKBOOK, cache_dir=OUTPUT_DIR/"cache", use_cache=USE_CACHE))
    trajectories.append(prepare_trajectory(raw, cfg, JUMP_SPEED_A_PER_PS))
inventory_table = inventory(trajectories)
display(inventory_table)
gap_tables = [discontinuities(r) for r in trajectories]
gap_table = pd.concat(gap_tables, ignore_index=True) if gap_tables else pd.DataFrame()
display(gap_table if not gap_table.empty else "No abrupt steps/source boundaries were flagged.")
save_table(inventory_table, f"{COMPARISON_PREFIX}_trajectory_inventory")
save_table(gap_table, f"{COMPARISON_PREFIX}_discontinuities")

## Temperature and energy: independent data source, separate figure

This cell runs even when the trajectory comes from XYZ/CSV. `auto` tries the combined temperature/energy CSV, then the ordered OUTCARs. A generic `Time` CSV header needs `THERMO_COLUMN_MAP={"time_ps":"Time", ...}` (or `time_fs` when the values are fs). A global Step column needs `THERMO_TIMESTEP_FS=0.5` after verifying POTIM. Missing/ambiguous input is reported in the source table; migration analysis remains available.

Temperature is shown above energy for each metal. `THERMO_ENERGY_REFERENCE="initial"` plots E−E(0); `"absolute"` retains absolute energy values. Exported thermo CSVs always retain **unshifted** energy. The source report records its original column/quantity. TOTEN from OUTCAR is electronic free energy, not the thermostat-extended conserved energy. Energy changes along a thermal trajectory are not NEB activation barriers.

Here E₀ in the axis label means the first saved energy, E(t=0). A negative difference means the recorded energy is below that reference. A generic `energy_eV` CSV does not establish which VASP energy quantity was extracted. In MD, OSZICAR **E** includes ionic kinetic and Nosé-thermostat terms; **F** is electronic free energy and **E0** is a distinct electronic-energy field. [VASP definitions](https://vasp.at/wiki/OSZICAR). A long downward trend therefore requires checking the original extraction and numerical behavior before assigning it to adsorption. To read electronic TOTEN directly, use `THERMO_SOURCE="outcar"` with `REFRESH_THERMODYNAMICS=True` on your storage drive.

In [ ]:
thermodynamics, thermo_rows = [], []
for cfg, trajectory in zip(RUNS, trajectories):
    try:
        if REPLOT_ONLY and not REFRESH_THERMODYNAMICS:
            thermo = load_saved_thermo(PREVIOUS_RESULTS_DIR, prefix_for(cfg["metal"]), cfg["metal"])
            print(f"{cfg['metal']}: using saved temperature/energy data")
        else:
            if REPLOT_ONLY and THERMO_SOURCE != "csv":
                layout = discover_run_folder(Path(cfg["thermo_csv"]).parent)
                cfg = {**cfg, "outcars": layout["outcars"], "missing_outcar_runs": layout["missing_outcar_runs"]}
            thermo = load_thermodynamics(cfg, trajectory, THERMO_SOURCE,
                                        OUTPUT_DIR/"cache", USE_CACHE)
    except (ValueError, FileNotFoundError) as exc:
        thermo_rows.append({"metal": cfg["metal"], "status": "unavailable", "details": str(exc)})
        print(str(exc))
        continue
    if thermo is None:
        thermo_rows.append({"metal": cfg["metal"], "status": "disabled"})
        continue
    thermodynamics.append(thermo)
    report = {"metal": cfg["metal"], "status": "loaded", "frames": len(thermo["time_ps"]),
              "duration_ps": float(np.ptp(thermo["time_ps"])),
              "trajectory_duration_ps": float(np.ptp(trajectory["time_ps"])),
              "energy_quantity": thermo["energy_quantity"], "source": str(thermo["source"]),
              "columns": str(thermo.get("column_mapping", "OUTCAR ionic-step records"))}
    thermo_rows.append(report)
    save_table(pd.DataFrame({"time_ps": thermo["time_ps"], "temperature_K": thermo["temperature_K"],
                             "energy_eV": thermo["energy_eV"], "source_segment": thermo["source_ids"]}),
               f"{prefix_for(cfg['metal'])}_temperature_energy_data")
    if SAVE_INDIVIDUAL_METALS:
        finish_figure(draw_thermodynamics([thermo], THERMO_ENERGY_REFERENCE, REFERENCE_TEMPERATURE_K),
                      f"{prefix_for(cfg['metal'])}_temperature_energy", SHOW_INDIVIDUAL_FIGURES)
thermo_source_table = pd.DataFrame(thermo_rows)
display(thermo_source_table)
save_table(thermo_source_table, f"{COMPARISON_PREFIX}_thermodynamic_sources")
if SAVE_COMPARISON_FIGURES:
    finish_figure(draw_thermodynamics(thermodynamics, THERMO_ENERGY_REFERENCE, REFERENCE_TEMPERATURE_K),
                  f"{COMPARISON_PREFIX}_temperature_energy")

# Record the full data ranges even when a narrower display range was requested.
range_rows = []
for r in thermodynamics:
    energy = r["energy_eV"]-r["energy_eV"][np.flatnonzero(np.isfinite(r["energy_eV"]))[0]]
    low, high = ENERGY_YLIMS[r["metal"]]
    range_rows.append({"metal": r["metal"], "energy_quantity": r["energy_quantity"],
                       "thermo_duration_ps": float(np.ptp(r["time_ps"])),
                       "delta_E_min_eV": float(np.nanmin(energy)), "delta_E_max_eV": float(np.nanmax(energy)),
                       "delta_E_final_eV": float(energy[-1]),
                       "temperature_mean_K": float(np.nanmean(r["temperature_K"])),
                       "temperature_min_K": float(np.nanmin(r["temperature_K"])),
                       "temperature_max_K": float(np.nanmax(r["temperature_K"])),
                       "energy_samples_outside_display": int(((energy<low)|(energy>high)).sum()),
                       "temperature_samples_outside_display": int(((r["temperature_K"]<TEMPERATURE_YLIM[0])|
                                                                       (r["temperature_K"]>TEMPERATURE_YLIM[1])).sum())})
range_table = pd.DataFrame(range_rows)
display(range_table)
save_table(range_table, f"{COMPARISON_PREFIX}_full_ranges_and_display_limits")

## Paths and vertical motion

The top-view migration figure shows time-colored paths over **green B and blue N sites**, with a circle at the start, square at the end, and maroon X at the vacancy reference. The lattice uses actual first-frame/POSCAR coordinates and is repeated across periodic images to cover the unwrapped trajectory. It is a **static reference background**; the h-BN sheet itself fluctuates during MD. Colors, dot opacity/size, path width, padding, and font scale are editable in the settings cell.

Separate height figures show sheet-relative height when substrate geometry is available; otherwise they show **z−z(0)**, which cannot identify passage through the h-BN plane. A three-N reference follows the local opening when full XYZ/OUTCAR coordinates are supplied. POSCAR-only references are static. Coordinate-displacement plots show x−x(0), y−y(0), and z−z(0). Input series use different initial-distance labels (Au 6nn, Ag/Cu 7nn); report those starting conditions in the caption rather than treating them as identical replicas.

Paths are broken at flagged jumps. Investigate those boundaries in the original OUTCAR/movie before treating the total time as one uninterrupted kinetic trajectory. Do not use trajectories from NEB images for MD time analysis.

The separate time bar is shared by all three migration figures. Changing each metal's individual normalization would make the same color refer to different elapsed times; the shared limits prevent that.

In [ ]:
# Every metal and the standalone bar use exactly the SAME time normalization.
last_time = max(float(r["time_ps"][-1]) for r in trajectories)
shared_time_limits = MIGRATION_TIME_LIMITS_PS or (0.0, float(np.ceil(last_time/10.0)*10.0))
if shared_time_limits[1] < last_time:
    raise ValueError("MIGRATION_TIME_LIMITS_PS does not cover the complete trajectories.")
migration_style = dict(show_lattice=SHOW_LATTICE, lattice_colors=LATTICE_COLORS,
                       lattice_alpha=LATTICE_ALPHA, lattice_dot_size=LATTICE_DOT_SIZE,
                       cmap=PATH_CMAP, padding_A=MIGRATION_PADDING_A, line_width=PATH_LINE_WIDTH,
                       time_limits_ps=shared_time_limits, show_colorbar=MIGRATION_INCLUDE_COLORBAR)
if MIGRATION_SAVE_COMPARISON:
    finish_figure(draw_migration_xy(trajectories, **migration_style), f"{COMPARISON_PREFIX}_migration_xy")
if SAVE_INDIVIDUAL_METALS:
    for r in trajectories:
        prefix = prefix_for(r["label"])
        finish_figure(draw_migration_xy([r], **migration_style), f"{prefix}_migration_xy", show=True)
        finish_figure(draw_height_history([r]), f"{prefix}_height_vs_time", SHOW_INDIVIDUAL_FIGURES)
        finish_figure(draw_coordinate_history([r]), f"{prefix}_xyz_displacements", SHOW_INDIVIDUAL_FIGURES)
finish_figure(draw_migration_timebar(shared_time_limits, PATH_CMAP), f"{COMPARISON_PREFIX}_migration_timebar")
if SAVE_COMPARISON_FIGURES:
    finish_figure(draw_height_history(trajectories), f"{COMPARISON_PREFIX}_height_vs_time")
    finish_figure(draw_coordinate_history(trajectories), f"{COMPARISON_PREFIX}_xyz_displacements")

## Mean-squared displacement and diffusion coefficient

Time-origin averaged in-plane MSD is

\[
\mathrm{MSD}_{xy}(\tau)=\langle[x(t+\tau)-x(t)]^2+[y(t+\tau)-y(t)]^2\rangle_t .
\]

For **stationary normal diffusion in two dimensions**, fit a linear lag-time interval:

\[
\mathrm{MSD}_{xy}(\tau)=4D_{xy}\tau+c.
\]

`ANALYSIS_WINDOWS_PS` chooses the absolute simulation interval; `DIFFUSION_FIT_LAG_PS` chooses the lag interval used for a fit. The default produces MSD plots but no diffusion fit. This matters for a metal moving toward a vacancy and then remaining trapped: the approach is nonstationary, and a confined late-time MSD plateaus rather than giving a bulk/free-surface diffusion coefficient. Inspect the log-log shape; ballistic motion has exponent near 2, normal diffusion near 1, and confinement approaches 0. A high linear-fit R² alone does not establish a valid D.

Do not infer an error bar from an ordinary regression of correlated MSD points. The optional nonoverlapping-block table reports descriptive variation, and reports nothing when the selected run is too short to support the chosen block/lag interval. Independent trajectories are preferable for uncertainty.

In the supplied run, the automatic continuous windows were Au 0–69.685 ps, Ag 11.7985–60.3185 ps, and Cu 10.1605–53.5105 ps. The Ag/Cu windows are after capture and their plateauing MSD reflects confined motion; Au includes approach plus trapped motion. These are not equivalent steady-diffusion windows. The saved diffusion summaries contained **no fitted D**. Time-origin-averaged MSD is already calculated, but an attempt frequency is not obtained from these curves or from the inverse of one capture time.

In [ ]:
msd_results, diffusion_fits, window_rows, block_tables = {}, {}, [], []
for r in trajectories:
    label = r["label"]
    msd, info, ids = msd_analysis(
        r, ANALYSIS_WINDOWS_PS.get(label, (None, None)), SEGMENT_IDS.get(label),
        MSD_STRIDE, MSD_MAX_LAG_PS)
    msd_results[label] = (msd, info, ids, r["metal"])
    window_rows.append(info)
    save_table(msd, f"{prefix_for(label)}_MSD_data")
    if label in DIFFUSION_FIT_LAG_PS:
        fit = fit_diffusion(msd, DIFFUSION_FIT_LAG_PS[label], r, ids)
        fit["label"] = label
        fit["L_for_tau_A"] = CHARACTERISTIC_LENGTH_A
        fit["tau_D_at_L_ps"] = characteristic_diffusion_time(fit["D_candidate_A2_ps"], CHARACTERISTIC_LENGTH_A)
        diffusion_fits[label] = fit
        blocks = block_diffusion_estimates(r, ids, DIFFUSION_FIT_LAG_PS[label], DIFFUSION_BLOCKS)
        if not blocks.empty:
            block_tables.append(blocks)
    if SAVE_INDIVIDUAL_METALS:
        finish_figure(draw_msd({label: msd_results[label]}, diffusion_fits),
                      f"{prefix_for(label)}_MSD", SHOW_INDIVIDUAL_FIGURES)

display(pd.DataFrame(window_rows))
save_table(pd.DataFrame(window_rows), f"{COMPARISON_PREFIX}_MSD_analysis_windows")
if SAVE_COMPARISON_FIGURES:
    finish_figure(draw_msd(msd_results, diffusion_fits), f"{COMPARISON_PREFIX}_MSD")
if diffusion_fits:
    diffusion_table = pd.DataFrame(diffusion_fits.values())
    display(diffusion_table)
    if block_tables:
        block_table = pd.concat(block_tables, ignore_index=True)
        display(block_table)
        save_table(block_table, f"{COMPARISON_PREFIX}_diffusion_nonoverlapping_blocks")
    else:
        print("No sufficiently long nonoverlapping blocks for the requested lag fit.")
else:
    print("No D fitted yet. Inspect the MSD and choose a stationary simulation window and a linear lag interval.")
diffusion_rows = []
for label, bundle in msd_results.items():
    info = bundle[1]
    row = {**info, "D_candidate_A2_ps": np.nan, "D_candidate_m2_s": np.nan,
           "D_candidate_cm2_s": np.nan, "tau_D_at_L_ps": np.nan,
           "assessment": "not fitted: select a stationary interval and lag range"}
    row.update(diffusion_fits.get(label, {}))
    diffusion_rows.append(row)
    save_table(pd.DataFrame([row]), f"{prefix_for(label)}_diffusion_summary")
diffusion_summary = pd.DataFrame(diffusion_rows)
save_table(diffusion_summary, f"{COMPARISON_PREFIX}_diffusion_summary")

## Compare confinement after capture

The original automatic windows include Au's approach but exclude Ag/Cu's approach. This additional figure compares **12–50 ps** for all three metals, a continuous interval after their persistent arrival near the vacancy. Its plateau amplitudes describe localized motion; they are not diffusion coefficients. The exact window and selected samples are exported. This comparison supplements the original MSD figure.


In [ ]:
post_capture_results, post_capture_windows, post_capture_summary = {}, [], []
if COMPARE_POST_CAPTURE_MSD:
    for r in trajectories:
        msd, info, ids = msd_analysis(r, POST_CAPTURE_WINDOW_PS, stride=MSD_STRIDE,
                                      max_lag_ps=min(MSD_MAX_LAG_PS, 9.0))
        required_duration = POST_CAPTURE_WINDOW_PS[1]-POST_CAPTURE_WINDOW_PS[0]
        if info["analysis_end_ps"]-info["analysis_start_ps"] < required_duration-0.01:
            raise ValueError("The chosen post-capture interval crosses a data gap or exceeds a trajectory. Select a continuous shared interval.")
        post_capture_results[r["label"]] = (msd, info, ids, r["metal"])
        post_capture_windows.append(info)
        plateau = msd[(msd.lag_ps>=1.0)&(msd.lag_ps<=9.0)]
        post_capture_summary.append({"metal": r["metal"],
                                     "mean_MSD_xy_A2_lag_1_9ps": plateau.MSD_xy_A2.mean(),
                                     "minimum_MSD_xy_A2_lag_1_9ps": plateau.MSD_xy_A2.min(),
                                     "maximum_MSD_xy_A2_lag_1_9ps": plateau.MSD_xy_A2.max()})
        save_table(msd, f"{prefix_for(r['label'])}_post_capture_MSD_data")
    finish_figure(draw_msd(post_capture_results), f"{COMPARISON_PREFIX}_MSD_post_capture")
    save_table(pd.DataFrame(post_capture_windows), f"{COMPARISON_PREFIX}_post_capture_MSD_windows")
    post_capture_summary = pd.DataFrame(post_capture_summary)
    save_table(post_capture_summary, f"{COMPARISON_PREFIX}_post_capture_MSD_summary")
    display(post_capture_summary)


## Vacancy capture, residence, and plane crossings

These answer different questions:

| Quantity | Meaning |
|---|---|
| First persistent entry | First entry within `CAPTURE_RADIUS_A`, followed by at least `CAPTURE_DWELL_PS` continuously inside. |
| Residence duration | Observed continuous time inside; a run ending or reaching a data gap while still inside supplies a lower bound, not a measured escape time. |
| Plane crossing | Change between persistently above and below the h-BN plane, with a deadband to reject small vibrations. Inspect the lateral vacancy distance at each event before identifying it as passage through a vacancy. |
| \(\tau_D(L)=L^2/(4D)\) | A length-dependent timescale for 2D normal diffusion; it is not an exponential capture or desorption lifetime. |
| Exponential capture \(\tau\) | Needs independently prepared, comparable replicas for the **same metal and starting-condition group**, including right-censored runs. One arrival per metal does not give a reliable rate constant. |

Coordinates from POSCAR allow a static vacancy reference. Full-coordinate XYZ/OUTCAR enables a moving local reference and is preferred for quantitative capture/vertical-motion analysis. Inspect automatically proposed N-neighbor indices. For pristine systems set `vacancy_neighbors=None`.

The radius, dwell time, plane-crossing deadband, and jump threshold are **editable analysis definitions chosen for this notebook**, not material constants taken from a paper. The optional capture-sensitivity table checks several radii and dwell times. Inspect that dependence before reporting a numerical capture time.

In [ ]:
capture_rows, crossings = [], []
for r in trajectories:
    capture_rows.append({"label": r["label"], "metal": r["metal"], "condition": r["condition"],
                         **capture_metrics(r, CAPTURE_RADIUS_A, CAPTURE_DWELL_PS)})
    events = crossing_events(r, PLANE_CROSSING_DEADBAND_A, PLANE_CROSSING_DWELL_PS)
    if not events.empty:
        crossings.append(events)
capture_table = pd.DataFrame(capture_rows)
display(capture_table)
save_table(capture_table, f"{COMPARISON_PREFIX}_capture_and_residence")
crossing_table = pd.concat(crossings, ignore_index=True) if crossings else pd.DataFrame()
display(crossing_table if not crossing_table.empty else "No plane-crossing events available/detected with the current geometry and thresholds.")
save_table(crossing_table, f"{COMPARISON_PREFIX}_plane_crossing_events")
if SAVE_COMPARISON_FIGURES:
    finish_figure(draw_vacancy_distances(trajectories, CAPTURE_RADIUS_A), f"{COMPARISON_PREFIX}_vacancy_distance")
for r in trajectories:
    label, prefix = r["label"], prefix_for(r["label"])
    save_table(capture_table[capture_table.label == label], f"{prefix}_capture_and_residence")
    if not crossing_table.empty:
        save_table(crossing_table[crossing_table.label == label], f"{prefix}_plane_crossing_events")
    if SAVE_INDIVIDUAL_METALS:
        finish_figure(draw_vacancy_distances([r], CAPTURE_RADIUS_A), f"{prefix}_vacancy_distance", SHOW_INDIVIDUAL_FIGURES)

if SHOW_CAPTURE_SENSITIVITY:
    sensitivity = []
    for r in trajectories:
        if r.get("distance_vacancy_A") is None:
            continue
        for radius in (1.0, 1.5, 2.0):
            for dwell in (0.25, 0.5, 1.0):
                metrics = capture_metrics(r, radius, dwell)
                sensitivity.append({"label": r["label"], "radius_A": radius, "dwell_ps": dwell,
                                    "first_persistent_entry_ps": metrics.get("first_persistent_entry_ps", np.nan),
                                    "status": metrics["capture_status"]})
    if sensitivity:
        sensitivity_table = pd.DataFrame(sensitivity)
        display(sensitivity_table)
        save_table(sensitivity_table, f"{COMPARISON_PREFIX}_capture_threshold_sensitivity")

# If you later have independent equivalent replicas, the helper is available:
# exponential_capture_tau(event_times_ps=[...], event_observed=[...])
# event_observed=False means right-censored; do not mix Au/Ag/Cu into one fit.

## Export the selected trajectories and settings

CSV exports preserve the full selected metal time series and segment identifiers, including raw coordinates. Smoothing/plot thinning is visual only; it does not replace raw coordinates or reconnect gaps. The input workbook, POSCARs, OUTCARs and movie files are never modified.

Examples under `AIMD_results`: `Au_bvac_6nn_migration_xy.png`, `Ag_bvac_7nn_temperature_energy.pdf`, `Cu_bvac_7nn_MSD_data.csv`, and `Au_Ag_Cu_bvac_capture_and_residence.csv`. PNG, SVG and PDF files contain the same figure. The diffusion summary is always saved; D fields remain blank until a fit is enabled and assessed. `output_file_index.csv` lists files produced by the current execution.

In [ ]:
if EXPORT_CSV:
    for r in trajectories:
        save_table(trajectory_table(r), f"{prefix_for(r['label'])}_trajectory_data")
settings = {"runs": RUNS, "workbook": WORKBOOK, "jump_speed_A_ps": JUMP_SPEED_A_PER_PS,
            "analysis_windows_ps": ANALYSIS_WINDOWS_PS, "segment_ids": SEGMENT_IDS,
            "msd_stride": MSD_STRIDE, "max_lag_ps": MSD_MAX_LAG_PS,
            "diffusion_fit_lag_ps": DIFFUSION_FIT_LAG_PS, "capture_radius_A": CAPTURE_RADIUS_A,
            "capture_dwell_ps": CAPTURE_DWELL_PS, "plane_deadband_A": PLANE_CROSSING_DEADBAND_A,
            "plane_dwell_ps": PLANE_CROSSING_DWELL_PS, "thermo_source": THERMO_SOURCE,
            "thermo_energy_reference": THERMO_ENERGY_REFERENCE,
            "reference_temperature_K": REFERENCE_TEMPERATURE_K,
            "font_scale": FONT_SCALE, "font_sizes": FONT_SIZES,
            "lattice_colors": LATTICE_COLORS, "lattice_alpha": LATTICE_ALPHA,
            "lattice_dot_size": LATTICE_DOT_SIZE, "path_cmap": PATH_CMAP,
            "path_line_width": PATH_LINE_WIDTH, "migration_padding_A": MIGRATION_PADDING_A,
            "line_width": LINE_WIDTH, "temperature_ylim": TEMPERATURE_YLIM, "energy_ylims": ENERGY_YLIMS,
            "migration_time_limits_ps": shared_time_limits, "replot_only": REPLOT_ONLY,
            "previous_results_dir": PREVIOUS_RESULTS_DIR, "post_capture_window_ps": POST_CAPTURE_WINDOW_PS}
settings_path = OUTPUT_DIR / f"{COMPARISON_PREFIX}_analysis_settings.json"
settings_path.write_text(json.dumps(settings, indent=2, default=str))
OUTPUT_FILES.append(settings_path)
output_index = pd.DataFrame({"filename": [p.name for p in OUTPUT_FILES],
                             "path": [str(p) for p in OUTPUT_FILES]})
output_index.to_csv(OUTPUT_DIR/"output_file_index.csv", index=False)
display(output_index)
print(f"Results: {OUTPUT_DIR.resolve()}")

## Optional animation

Set `ANIMATE_LABEL="Cu"` (or another run label) and rerun this cell. The animation uses the already extracted metal trajectory. Saving a GIF uses Pillow; the initial 1 GB movie stays on your storage drive. The animation trail resets across flagged gaps.

In [ ]:
if ANIMATE_LABEL is not None:
    selected = next(r for r in trajectories if r["label"] == ANIMATE_LABEL)
    animation = make_xy_animation(selected)
    if SAVE_ANIMATION_GIF:
        animation.save(OUTPUT_DIR/f"{prefix_for(ANIMATE_LABEL)}_trajectory.gif", writer="pillow", fps=25)
    from IPython.display import HTML
    display(HTML(animation.to_jshtml()))
    plt.close(animation._fig)

## Published methods, related applications, and software references

This is custom postprocessing code. It is not copied from a publication and does not reproduce any paper's entire analysis package. The initial version used standard Einstein-MSD definitions and software documentation. The MoS2 study provided physical/visual context. The following verified journal references are now included to support and delimit the analysis:

| Published reference | Role here |
|---|---|
| Zarshenas, Sangiovanni & Sarakinos, **Diffusion and magnetization of metal adatoms on single-layer molybdenum disulfide at elevated temperatures**, *JVST A* 42, 023409 (2024). [DOI](https://doi.org/10.1116/6.0003207) | Closely related Au/Ag/Cu AIMD application and trajectory comparison; not a source of h-BN capture thresholds. |
| Maginn et al., **Best Practices for Computing Transport Properties 1. Self-Diffusivity and Viscosity from Equilibrium Molecular Dynamics**, *Living J. Comput. Mol. Sci.* 1, 6324 (published 2018). [DOI](https://doi.org/10.33011/livecoms.1.1.6324) | Equilibrium diffusion/MSD methodology, sampling, reproducibility, and transport-property analysis. |
| Bullerjahn, von Bülow & Hummer, **Optimal estimates of self-diffusion coefficients from molecular dynamics simulations**, *J. Chem. Phys.* 153, 024116 (2020). [DOI](https://doi.org/10.1063/5.0008312) | Correlated MSD points, uncertainty and testing diffusive behavior. This notebook's candidate straight-line fit is simpler; it does not implement their full correlated estimator. |
| Pranami & Lamm, **Estimating Error in Diffusion Coefficients Derived from Molecular Dynamics Simulations**, *J. Chem. Theory Comput.* 11, 4586–4592 (2015). [DOI](https://doi.org/10.1021/acs.jctc.5b00574) | Why independent simulations and careful uncertainty analysis matter. Restart parts are not independent replicas. |
| Ghaderzadeh et al., **Platinum Atoms Dynamics on the Surface of Hexagonal Boron Nitride Containing Vacancy Defects**, *ACS Appl. Mater. Interfaces* 18, 9216–9224 (2026). [DOI](https://doi.org/10.1021/acsami.5c22977) | Related pristine/defective h-BN AIMD and vacancy anchoring; a physical comparison, not the algorithm used here. |

Software/file-format documentation:

- [VASP POTIM: MD time step in femtoseconds](https://vasp.at/wiki/POTIM)
- [VASP XDATCAR and NBLOCK: saved-frame spacing](https://vasp.at/wiki/XDATCAR)
- [LAMMPS: unwrapped MSD and center-of-mass drift](https://docs.lammps.org/latest/compute_msd.html)
- [GROMACS: MSD fitting, lag selection and sampling limitations](https://manual.gromacs.org/documentation/current/onlinehelp/gmx-msd.html)
- [VS Code: local Jupyter notebook kernels](https://code.visualstudio.com/docs/datascience/jupyter-notebooks)

Tests of this notebook use the supplied spreadsheet and small synthetic format/geometry checks. Your unuploaded OUTCAR/movie files must be checked locally using the inventory, atom ordering, cell and discontinuity diagnostics. Do not infer activation energies or an Arrhenius prefactor from one temperature alone.

### Attempt frequencies for a chosen NEB hop

Harmonic transition-state theory uses `k(T)=nu_star*exp(-Ea/(kB*T))`. Its Vineyard prefactor is the product of the positive minimum-state vibrational frequencies divided by the product of the positive saddle-state frequencies. The unstable saddle mode is excluded; consistently treat fixed and zero modes. This requires minimum/saddle dynamical matrices (additional vibrational DFT calculations if they do not already exist), not just NEB energies. [Vineyard, J. Phys. Chem. Solids 3, 121–127 (1957)](https://doi.org/10.1016/0022-3697(57)90059-8); [QuantumATK HTST documentation](https://docs.quantumatk.com/manual/Types/HTSTEvent/HTSTEvent.html). These prefactors have not been calculated by this notebook.